# Baseline models and their feature selection

**Original file:** `1_Spectors/1_JypiterLabProcessing/01_SpectralAnalysis_Part00.ipynb` in the author's project archive.

Fits the bootstrap PLSR models for 2014, 2017 and all dates, draws Figure 2 and the reference panels of Figures 6 and 8, predicts at pixel level and repeats the models with the bands selected through VIP. The equivalent code in this repository is in `src/model_transfer/baseline.py` and `feature_selection.py`.

**Status:** kept for reference; it does not run as it is. It needs Python 3.6, library versions of that time, files that are not shared and, in places, manual changes between runs. `<PROJECT_ROOT>`, `<RAW_DATA>` and `<CONDA_ENVS>` replace local folders. To reproduce the results, use the scripts in the main folder of this repository.

In [ ]:
# Switches for tasks that were turned off in the saved notebook. Leave them off to read saved files.
SAVE_OUTPUTS = False

## 1. Setup

### Packages

Libraries and working folder. *(Original cell 1.)*

In [ ]:
import os, sys
import copy
import fiona
os.environ['GDAL_DATA'] = r'<CONDA_ENVS>\Python365_PCSE\Library\share\gdal'
import gdal
import gdalconst
import geopandas as gpd
import glob
import itertools
import matplotlib
import matplotlib.pyplot as plt
%matplotlib qt
import numpy as np
np.set_printoptions(suppress=True)
import ogr
import osr
import pandas as pd
import pickle
import progressbar as pbar
import pylab
os.environ['PROJ_LIB'] = r'<CONDA_ENVS>\Python365\Library\share\proj'
import pyproj
import random
import scipy
import scipy.stats

import affine
import rasterio

import sklearn
from   sklearn.cross_decomposition import PLSRegression
import sklearn.cluster
import sklearn.decomposition
from   sklearn.model_selection import GridSearchCV, cross_val_score, KFold, RepeatedKFold
from   sklearn.pipeline import make_pipeline, Pipeline
from   sklearn.preprocessing import StandardScaler, RobustScaler

import spectral

print("This notebook was built with:")
print("python version: %s " % sys.version)

## 2. Data preparation

### Extracted spectra

Reads the spectra produced by notebook 01. *(Original cell 2.)*

In [ ]:
# 2014 and 2017 - after buffering
os.chdir(r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV') # get folder with images - acq 1
compRefT = 'New_UpD.'  # latest file version

PatternUpload = '*SU_VegProp_B_FLD_' + compRefT + '*pkl' # nearest flight line to the plot center
DataExtAll = glob.glob(PatternUpload)
SpecAll = []
for ii in range(len(DataExtAll)):  # acq
    SpecAll.append( pd.read_pickle(DataExtAll[ii]) )

# column names - spectral information
col_ref  = [ 'b'+ [str(x) if x>=100 else ( '00' + str(x) if x<10 else '0'+str(x) )][0] for x in range(0, 101, 1) ]

# get only spectral information ; 0-1 or 0-100 interval
SpecAll_sel_0_1   = []
SpecAll_sel_0_100 = []
for ii in range(len(SpecAll)): # acq
    SpecAll_sel_0_1.append(   np.expand_dims(SpecAll[ii].loc[:,col_ref] / 10000, axis=0) )
    SpecAll_sel_0_100.append( np.expand_dims(SpecAll[ii].loc[:,col_ref] / 100, axis=0) )

# 2014 and 2017 - Full dataset - before buffering
PatternUpload  = '*SU_VegProp_FLD_' + compRefT +'*pkl' # nearest flight line to the plot center
DataExtAllFull = glob.glob(PatternUpload)
SpecAllFull = []
for ii in range(len(DataExtAllFull)):  # acq
    SpecAllFull.append( pd.read_pickle(DataExtAllFull[ii]) )

# get only spectral information ; 0-1 or 0-100 interval
SpecAllFull_sel_0_1   = []
SpecAllFull_sel_0_100 = []
for ii in range(len(SpecAllFull)): # acq
    SpecAllFull_sel_0_1.append(   np.expand_dims(SpecAllFull[ii].loc[:,col_ref] / 10000, axis=0) )
    SpecAllFull_sel_0_100.append( np.expand_dims(SpecAllFull[ii].loc[:,col_ref] / 100, axis=0) )

### Spatial coordinates

Adds the coordinates of each pixel. *(Original cell 3.)*

In [ ]:
os.chdir(r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV') # get folder with images - acq 1
PatternUpload  = '*SU_geo_VegProp_B_FLD_' + compRefT + '*pkl' # nearest flight line to the plot center
DataExt_geoAll = glob.glob(PatternUpload)

# coord. syst.
crsT = 'EPSG:32632'
projRefT_wkt = pyproj.crs.CRS(crsT).to_wkt()

SpecAll_geo = []
for ii in range(len(DataExt_geoAll)):  # acq
    SpecAll_geo.append( gpd.GeoDataFrame( data=pd.DataFrame( pd.read_pickle(DataExt_geoAll[ii]).drop(['geometry'], axis=1) ), \
                                          crs=projRefT_wkt, \
                                          geometry=pd.read_pickle(DataExt_geoAll[ii]).geometry.tolist() ) )

# get points with spatial coordinates  -  Full spectra
PatternUpload      = '*SU_geo_VegProp_FLD_' + compRefT + '*pkl' # nearest flight line to the plot center
DataExt_geoAllFull = glob.glob(PatternUpload)

SpecAll_geoFull = []
for ii in range(len(DataExt_geoAllFull)):  # acq
    SpecAll_geoFull.append( gpd.GeoDataFrame( data=pd.DataFrame( pd.read_pickle(DataExt_geoAllFull[ii]).drop(['geometry'], axis=1) ), \
                                              crs=projRefT_wkt, \
                                              geometry=pd.read_pickle(DataExt_geoAllFull[ii]).geometry.tolist() ) )

### Spectral smoothing

Resamples the spectra to a bandwidth of 30 nm and removes the last two bands. *(Original cell 4.)*

In [ ]:
# if wanted >> resample spectra - https://github.com/spectralpython/spectral/blob/master/spectral/algorithms/resampling.py

# number of bands to consider in the 'smoothing window'
newFWHM = 30

# resampler - without last 2 bands (attention: original dataset should be resized >> nbr of bands should match the resampler -
FWHM_origin = [5] * len(range(450, 941, 5))
FWHM_transf = [newFWHM] * len(range(450, 941, 5))
center_origin = range(450, 941, 5)

# resampler for raw >> smoothed conversion
resamplerO = spectral.BandResampler(centers1=center_origin, fwhm1=FWHM_origin, centers2=center_origin,
                                    fwhm2=FWHM_transf)

SpecAll_sel_0_1_res = copy.deepcopy(SpecAll_sel_0_1)
for ii in range(len(SpecAll_sel_0_1_res)):  # acq
    SpecAll_sel_0_1_res[ii] =  SpecAll_sel_0_1_res[ii][:,:,:len(center_origin)]
    for jj in range(SpecAll_sel_0_1[ii].shape[1]):  # pxls
        SpecAll_sel_0_1_res[ii][0,jj,:] = resamplerO( SpecAll_sel_0_1[ii][0,jj,:len(center_origin)] )  # resample

# get 0-100 interval dataset - smoothed spectra
SpecAll_sel_0_100_res = copy.deepcopy(SpecAll_sel_0_100)
for ii in range(len(SpecAll_sel_0_1_res)):   # date
    SpecAll_sel_0_100_res[ii] = SpecAll_sel_0_1_res[ii] * 100

# resample full dataset used for predictions
SpecAllFull_sel_0_1_res = copy.deepcopy(SpecAllFull_sel_0_1)
for ii in range(len(SpecAllFull_sel_0_1_res)):  # acq
    SpecAllFull_sel_0_1_res[ii] =  SpecAllFull_sel_0_1_res[ii][:,:,:len(center_origin)]
    for jj in range(SpecAllFull_sel_0_1[ii].shape[1]):  # pxls..
        SpecAllFull_sel_0_1_res[ii][0,jj,:] = resamplerO( SpecAllFull_sel_0_1[ii][0,jj,:len(center_origin)] )  # resample

# get 0-100 interval dataset for smoothed spectra
SpecAllFull_sel_0_100_res = copy.deepcopy(SpecAllFull_sel_0_100)
for ii in range(len(SpecAllFull_sel_0_1_res)):   # date
    SpecAllFull_sel_0_100_res[ii] = SpecAllFull_sel_0_1_res[ii] * 100

### Ground truth

Reads the laboratory measurements of the traits. *(Original cell 5.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\2_Ground_aux' )

# fields - long format
ToSRefT = [ 'Location', 'Year', 'Date', 'Treatment', 'Name', 'Replicate', \
            'Variable', 'Value', 'DateMonth', 'DateDay' ]

# load - Kleve | 2012 to 2019
HarvestKleveRaw = pd.read_csv( '568_2012-2017_2019_Kleve_All_Merge_New_Adjusted' + '.csv', sep=';')

# rearrange in wide format

metaRefT = [ 'Location', 'Date', 'Year', 'DateMonth', 'DateDay', \
             'Treatment', 'Name', 'Replicate', 'Cut' ]

# reference to variables
varRefT = [ 'FM_kg_plot', 'DM_dt_ha', 'CanHeight_cm', 'CrudeAsh_perc_DM', \
            'CrudeFiber_perc_DM', 'CrudeProt_perc_DM', 'N_perc_DM' ]

# reference to dates

Gdata_df_all = []
for ii in range( len(varRefT) ):  # trait
    # dates with observations for a given trait
    dataToPSelT = HarvestKleveRaw[ (HarvestKleveRaw.Variable == varRefT[ii]) ]
    dataToPSelT = dataToPSelT.reset_index(drop=True)
    Gdata_df_allT = dataToPSelT.loc[:, metaRefT + ['Value'] ].copy(deep=True)
    Gdata_df_allT.columns =  metaRefT + [ varRefT[ii] ]
    if len(Gdata_df_all) == 0:
        Gdata_df_all =  Gdata_df_allT.copy(deep=True)
    else:
        Gdata_df_all[ varRefT[ii] ] = Gdata_df_allT[ varRefT[ii] ]

### Average per plot

Combines spectra and traits at plot level, used for dry matter. *(Original cell 6.)*

In [ ]:
# subset data for each date
HarvRef  = [ '20140515', '20141015', '20170510', '20170830', '20171026' ]

for ii in range(len(SpecAll_sel_0_1_res)):

    TreatRefAllT = [ (SpecAll[ii].TreatCode.tolist()[val]).split('.')[1] for val in range(SpecAll[ii].shape[0]) ]
    TreatRefT    = np.unique(TreatRefAllT).tolist()
    for kk in pbar.progressbar( range(len( TreatRefT )) ):  # treat

        RepRefT =  sorted( SpecAll[ii].loc[ (np.array(TreatRefAllT) == TreatRefT[kk]), :].Replicate.unique().tolist() )
        for jj in range(len(RepRefT)):  # replicate

            MetaST = SpecAll[ii].loc[ (SpecAll[ii].Replicate == RepRefT[jj]) & (np.array(TreatRefAllT) == TreatRefT[kk]), :]
            SpecAll_sel_0_1_res_dfT = pd.DataFrame( SpecAll_sel_0_1_res[ii][0,:,:] )

            SpecAll_sel_0_1_res_dfT_sel  = pd.DataFrame( SpecAll_sel_0_1_res_dfT.iloc[MetaST.index.tolist(), :].mean().values.reshape((1, SpecAll_sel_0_1_res_dfT.shape[1])) )
            SpecAll_sel_0_1_res_dfT_meta = pd.DataFrame( np.array( (HarvRef[ii], TreatRefT[kk], RepRefT[jj]) ).reshape((1, len((HarvRef[ii], TreatRefT[kk], RepRefT[jj])))) )

            GdataT = Gdata_df_all.loc[ (Gdata_df_all.Date == (HarvRef[ii][0:4] + '-' + HarvRef[ii][4:6] + '-' + HarvRef[ii][6:]) ) & (Gdata_df_all.Replicate == RepRefT[jj]) & \
                                       (Gdata_df_all.Treatment == int( TreatRefT[kk] )), [ 'DM_dt_ha', 'N_perc_DM', 'CrudeProt_perc_DM']]

            if all([(ii == 0), (jj == 0), (kk == 0)]):
                SpecAll_sel_0_1_res_Avg = pd.concat( [SpecAll_sel_0_1_res_dfT_meta.reset_index(drop=True), GdataT.reset_index(drop=True), \
                                                      SpecAll_sel_0_1_res_dfT_sel.reset_index(drop=True)], \
                                                      axis=1, ignore_index=True, sort=False )
                col_ref = ['Date', 'Treatment', 'Replicate'] + ['DM_dt_ha', 'N_perc_DM', 'CrudeProt_perc_DM'] + \
                          list( range(SpecAll_sel_0_1_res_dfT_sel.shape[1]) )
                SpecAll_sel_0_1_res_Avg.columns = col_ref

            else:
                SpecAll_sel_0_1_res_AvgT = pd.concat( [SpecAll_sel_0_1_res_dfT_meta.reset_index(drop=True), \
                                                       GdataT.reset_index(drop=True), SpecAll_sel_0_1_res_dfT_sel.reset_index(drop=True)], \
                                                       axis=1, ignore_index=True, sort=False )
                col_ref = ['Date', 'Treatment', 'Replicate'] + ['DM_dt_ha', 'N_perc_DM', 'CrudeProt_perc_DM'] + \
                          list( range(SpecAll_sel_0_1_res_dfT_sel.shape[1]) )
                SpecAll_sel_0_1_res_AvgT.columns = col_ref
                SpecAll_sel_0_1_res_Avg = SpecAll_sel_0_1_res_Avg.append(SpecAll_sel_0_1_res_AvgT, ignore_index=True, sort=False)

# add year
SpecAll_sel_0_1_res_Avg.loc[:,'Year']    = [ SpecAll_sel_0_1_res_Avg.Date[val][:4] for val in range(len(SpecAll_sel_0_1_res_Avg.Date)) ]

# add N content per ha (canopy level) - normally area based (derived using LAI, not possible in our case)
SpecAll_sel_0_1_res_Avg.loc[:,'N_kg_ha'] = (SpecAll_sel_0_1_res_Avg.DM_dt_ha * (SpecAll_sel_0_1_res_Avg.N_perc_DM/100)).tolist()
SpecAll_sel_0_1_res_Avg.loc[:,'N_kg_ha'] = SpecAll_sel_0_1_res_Avg.N_kg_ha * 100  # transform dt in kg
SpecAll_sel_0_1_res_Avg.loc[:,'CrudeProt_kg_ha'] = (SpecAll_sel_0_1_res_Avg.DM_dt_ha * (SpecAll_sel_0_1_res_Avg.CrudeProt_perc_DM/100)).tolist()
SpecAll_sel_0_1_res_Avg.loc[:,'CrudeProt_kg_ha'] = SpecAll_sel_0_1_res_Avg.CrudeProt_kg_ha * 100  # transform dt in kg

# get dataset in the right order
MetaT = ['Date', 'Year', 'Treatment', 'Replicate', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha']
NbrBands = SpecAll_sel_0_1_res_Avg.shape[1] - len(MetaT)
SpecAll_sel_0_1_res_Avg = SpecAll_sel_0_1_res_Avg.loc[:, MetaT+list(range(NbrBands)) ]

# select only spectra

### Average per treatment

Combines spectra and traits at treatment level, used for the N traits. *(Original cell 7.)*

In [ ]:
DataRef    = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
colRef     = ['Date', 'Year', 'Treatment', 'Replicate', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha']
colRefRed  = ['Date', 'Treatment', 'Year', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha']

for ii in range(len(DataRef)):  # date
    TreatRef = sorted( SpecAll_sel_0_1_res_Avg.loc[SpecAll_sel_0_1_res_Avg.Date == DataRef[ii], :].Treatment.unique() )

    for jj in pbar.progressbar( range(len(TreatRef)) ):  # treat
        DataSelT = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Date  == DataRef[ii]) & \
                                                (SpecAll_sel_0_1_res_Avg.Treatment == TreatRef[jj]), :]
        DataSelT1 = DataSelT.loc[:, ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha'] ].mean().values
        DataSelT2 = DataSelT.loc[:, range(SpecAll_sel_0_1_res_Avg.shape[1]-len(colRef)) ].mean().values
        dict1 = dict( enumerate( np.transpose(DataSelT2) ) )
        dict2 = { (list(range(SpecAll_sel_0_1_res_Avg.shape[1]-len(colRef))))[x]: dict1[x] for x in dict1.keys() }

        dict2['Date'] = DataRef[ii]
        dict2['Treatment'] = TreatRef[jj]
        dict2['Year'] = int(str(DataRef[ii])[:4])
        dict2['DM_dt_ha'] = DataSelT1[0]
        dict2['N_perc_DM'] = DataSelT1[1]
        dict2['N_kg_ha'] = DataSelT1[2]
        dict2['CrudeProt_perc_DM'] = DataSelT1[3]
        dict2['CrudeProt_kg_ha'] = DataSelT1[4]
        DataSel_temp = pd.DataFrame( dict2, index=range(1), columns=colRefRed+list(range(SpecAll_sel_0_1_res_Avg.shape[1]-len(colRef))) )
        if all([(ii==0), (jj==0)]):
            SpecAll_sel_0_1_res_Avg_treat = DataSel_temp.copy(deep=True)
        else:
            SpecAll_sel_0_1_res_Avg_treat = SpecAll_sel_0_1_res_Avg_treat.append( DataSel_temp, ignore_index=True )

## 3. Baseline models

### Bootstrap PLSR

Fits 100 models per dataset and trait, choosing the number of factors with out-of-bag samples, and predicts the samples of the other year and the pixels. *(Original cell 8.)*

In [ ]:
# Variable Importance in Projection - formulation for VIP calculation   # https://github.com/scikit-learn/scikit-learn/issues/7050
def vip(model):
    t = model.x_scores_
    w = model.x_weights_
    q = model.y_loadings_
    p, h = w.shape
    vips = np.zeros((p,))
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)
    for i in range(p):
        weight  = np.array([ (w[i,j] / np.linalg.norm(w[:,j]))**2 for j in range(h) ])
        vips[i] = np.sqrt(p*(s.T @ weight)/total_s)
    return vips

# max and min nbr of factors to use
nbrFactMax_thrshld = 10
nbrFactMin = 3

# model fitting - PLSR
TraitRef  = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]  # reduced version of the traits - since crude protein seems to be estimated from N content
TraitRefL = ['Dry matter [dt/ha]', 'N [%]', 'N [kg/ha]']

DateRef  = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = np.unique( [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ] ).tolist()

# get seed for training and test sets selection - calib and valid

seedAll = [325685, 73256841, 568]

# for CV repetition
BootSNbr   = 100 # nbr of resampling | repetitions
seedRepTmF = []
np.random.seed(seed=23577)
for elmnt in range(len(TraitRef)):
    seedRepTmF.append( np.random.uniform(low=0, high=1000000000, size=BootSNbr) )
    seedRepTmF[-1] = [ int(round(seedRepTmF[-1][val])) for val in range(len(seedRepTmF[-1])) ]

colRef     = ['Date', 'Year', 'Treatment', 'Replicate', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha']
colRefRed  = ['Date', 'Treatment', 'Year', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha']

# nbr of bands with data
waveRef    = SpecAll_sel_0_1_res_Avg.shape[1] - len(colRef)
waveRefRed = SpecAll_sel_0_1_res_Avg_treat.shape[1] - (len(colRefRed) + ( np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1].max() - \
                                                                         (np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1].min()-1) if \
                                                                         len(np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1]) > 0 else 0 ))  # last part >> remove NaN values

# indicate if dataset should be scaled before model fitting
ScaleOpt = True

# model fitting - HYMSY hyper data
CoefBooF_all   = []
InterBooF_all  = []
PLSRFactF_all  = []
R2EvalF_all    = []
RMSEEvalF_all  = []
R2ValF_all     = []
RMSEValF_all   = []
TestUnF_all    = []
ValiUnF_all    = []
OutDUnF_all    = []
VIPOUnF_all    = []
PredMapF_all   = []
sampleValF_all = []
sampleBooF_all = []
for ii in range( (len(DateRefY)+1) ):  # dataset (year) + combining all dates
    if ii in [0,1]:
        PeriodRef = [ DateRefY[ii] ]
    else:  # combining all dates
        PeriodRef = DateRefY
    CoefBooF   = []
    InterBooF  = []
    PLSRFactF  = []
    R2EvalF    = []
    RMSEEvalF  = []
    R2ValF     = []
    RMSEValF   = []
    TestUnF    = []
    ValiUnF    = []
    OutDUnF    = []
    VIPOUnF    = []
    PredMapF   = []
    sampleValF = []
    sampleBooF = []
    for jj in range(len(TraitRef)):  # trait
        if( jj == 0 ):  # dataset to consider - average at plot level - Dry mass
            DataP  = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
            GroupF = [ str(DataP.Replicate.tolist()[val]) + '_' + str(DataP.Treatment.tolist()[val]) for val in range(DataP.shape[0]) ]
            if( ii in [0,1] ):  # dataset left outside
                DataPOut  = SpecAll_sel_0_1_res_Avg.loc[ ~(SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
            else:
                DataPOut  = []
        else:  # dataset to consider - average at treat level - Nitrogen
            DataP  = SpecAll_sel_0_1_res_Avg_treat.loc[ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
            GroupF = DataP.Treatment.tolist()
            if ii in [0,1]:  # dataset left outside
                DataPOut  = SpecAll_sel_0_1_res_Avg_treat.loc[~ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
            else:
                DataPOut  = []

        # reduce dataset to predictive and dependent variables only
        DataM_df = DataP.loc[ :, (DataP.columns == TraitRef[jj]) | DataP.columns.isin(range(waveRefRed)) ]  # using always the reduced data
        BRef     = [ 'b' + str(range(waveRefRed)[val]) if range(waveRefRed)[val] > 9 else 'b0' + str(range(waveRefRed)[val]) for val in range(waveRefRed) ]
        DataM_df.columns = ['Trait'] + BRef
        DataM_df = DataM_df.reset_index(drop=True)
        if( ii in [0,1] ):  # dataset left outside
            DataPOut_df = DataPOut.loc[ :, (DataPOut.columns == TraitRef[jj]) | DataPOut.columns.isin(range(waveRefRed)) ]
            DataPOut_df.columns = ['Trait'] + BRef
            DataPOut_df = DataPOut_df.reset_index(drop=True)

        # calib and valid datasets - 0.7/0.3
        gsTV = sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seedAll[ii])  # random nbr associated with dataset|year since this is the fctor that matters ;
                                                                                                               # e.g., for both N related traits the same samples are drawn for the same datatset|year
        trainIF_Oall = []
        testIF_Oall  = []
        for trainI, testI in gsTV.split(DataM_df.loc[:,BRef], DataM_df.Trait, GroupF):
            trainIF_Oall.append( trainI )
            testIF_Oall.append( testI )
        trainIF_Oall = trainIF_Oall[0].tolist()
        testIF_Oall  = testIF_Oall[0].tolist()

        # save reference to validation set - attention: based on row nbr not index
        sampleValF.append( testIF_Oall )

        # get calibration and validation sets
        Cal_df   = DataM_df.iloc[ trainIF_Oall,: ]
        GroupFTr = [ GroupF[val] for val in range(len(GroupF)) if val in trainIF_Oall ]
        Val_df   = DataM_df.iloc[ testIF_Oall,: ]
        GroupFTt = [ GroupF[val] for val in range(len(GroupF)) if val in testIF_Oall ]

        # Create arrays were to store predictions
        TestUn = np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan )  # samples in the calib x repetitions (boot)
        EvalUn = np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan )  # samples in the calib x repetitions (boot)
        ValiUn = np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan )
        if ii in [0,1]:
            OutDUn = np.full( (DataPOut_df.shape[0], len(seedRepTmF[jj])), np.nan )
        VIPOUn = np.full( (len(BRef), len(seedRepTmF[jj])), np.nan )
        CoefUn = np.full( (len(BRef), len(seedRepTmF[jj])), np.nan )

        # Create array were to store predictions - prediction maps
        PredMapT = []
        # for val in range(len(SpecAll_sel_0_1_res)):  # acq
        for val in range( len(SpecAllFull_sel_0_1_res) ):  # acq
            PredMapT.append( np.full( (SpecAllFull_sel_0_1_res[val].shape[1], len(seedRepTmF[jj])), np.nan ) )

        # 'internal' repetition - bootstrap
        InterBooT  = []
        PLSRFactT  = []
        R2EvalT    = []
        RMSEEvalT  = []
        R2ValT     = []
        RMSEValT   = []
        sampleBooT = []
        for ll in pbar.progressbar( range(len(seedRepTmF[jj])) ):  # repetition time of bootstrap

            # get model fitting and test sets
            Cal_dfT   = Cal_df.copy(deep=True)
            GroupFTrT = copy.deepcopy(GroupFTr)
            sizeRef   =  int( (Cal_dfT.shape[0]*0.3333) / len(DataP.Date.unique().tolist()) )
            # sizeRef >> divide by nbr of acquistions since its grouped by this factor, therefore selected samples will be multiplied by this factor
            gsBoo   = sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size=sizeRef, random_state=seedRepTmF[jj][ll])
            for trainI, testI in gsBoo.split(Cal_dfT.loc[:,BRef], Cal_dfT.Trait, GroupFTrT):
                testIF    = testI.tolist()
                testIFidx = Cal_df.loc[ Cal_dfT.iloc[testIF,:].index.tolist(), :].index.tolist()  # index needed to be used with drop

            # save reference to test set within each boot repetition - attention: based on row nbr not index
            sampleBooT.append( testIF )

            # update datasets
            Cal_dfT      = Cal_df.drop( labels = testIFidx )  # calibration set for the bootstrap repetition
            GroupFTrTidx = [ val for val in range(Cal_df.shape[0]) if Cal_df.index.tolist()[val] not in testIFidx ]
            GroupFTrT    = [ GroupFTr[val] for val in GroupFTrTidx ]
            Eval_dfT     = Cal_df.loc[ testIFidx, :]          # test set for the bootstrap repetition

            # fit PLSR model to calibration set corresponding to the bootstrap repetition
            RMSETO = []
            R2TO   = []
            nbrFactMax = np.min( [ min(Cal_dfT.shape), nbrFactMax_thrshld ] )  # maximun nbr of PLS factors should be <= nbr of samples
            for mm in range( (nbrFactMax-nbrFactMin)+1 ):  # nbr of PLS factors to try
                plsM      = PLSRegression( n_components=range(nbrFactMin, nbrFactMax+1, 1)[mm], scale=ScaleOpt )
                plsM.fit( Cal_dfT.loc[:,BRef].values,  Cal_dfT.loc[:,'Trait'].values.reshape( (Cal_dfT.loc[:,'Trait'].shape[0],1) ) )
                PredEvalO = [ plsM.predict(Eval_dfT.loc[:,BRef].values).tolist()[val][0] for val in range(len(plsM.predict(Eval_dfT.loc[:,BRef].values).tolist())) ]
                ResT =  Eval_dfT.Trait - np.array(PredEvalO)
                uVal = (ResT**2).sum()
                vVal = ((Eval_dfT.Trait.values.reshape(-1, 1) - Eval_dfT.Trait.values.reshape(-1, 1).mean())**2).sum()
                R2TO.append( (1-(uVal/vVal)) )
                RMSETO.append( np.sqrt( uVal/Eval_dfT.shape[0] ) )

            # selected 'optimal' nbr of factors for boot repetition
            nbrFactSel = list(range(nbrFactMin, nbrFactMax+1, 1))[ np.where( (RMSETO == np.array(RMSETO).min()) )[0][0] ]
            PLSRFactT.append( nbrFactSel )
            plsM = PLSRegression( n_components=nbrFactSel, scale=ScaleOpt )
            plsM.fit( Cal_dfT.loc[:,BRef].values,  Cal_dfT.loc[:,'Trait'].values.reshape( (Cal_dfT.loc[:,'Trait'].shape[0],1) ) )

            # get predictions and model parameters
            TestUn[Cal_dfT.index.tolist(),ll] = plsM.predict(Cal_dfT.loc[:,BRef].values).reshape(len(Cal_dfT.index.tolist()),)
            EvalUn[testIFidx,ll] =  plsM.predict(Eval_dfT.loc[:,BRef].values).reshape(len(testIFidx),)
            ValiUn[Val_df.index.tolist(),ll]  = plsM.predict(Val_df.loc[:,BRef].values).reshape(len(Val_df.index.tolist()),)
            if ii in [0,1]:
                OutDUn[:,ll] = plsM.predict( DataPOut_df.loc[:,BRef].values ).reshape(DataPOut_df.shape[0],)
            VIPOUn[:,ll] = vip(plsM)
            CoefUn[:,ll] = [ plsM.coef_.tolist()[val][0] for val in range(len(plsM.coef_.tolist())) ]
            InterBooT.append( (plsM._y_mean - np.dot(plsM._x_mean , plsM.coef_)).tolist()[0] )  # https://stackoverflow.com/questions/41188410/how-obtain-the-intercept-of-the-pls-regression-sklearn

            # R2 and RMSE for calib set
            PredT = copy.deepcopy( EvalUn[testIFidx,ll] )
            ResT  = Eval_dfT.Trait.values - np.array( PredT )
            uVal = (ResT**2).sum()
            vVal = ((Eval_dfT.Trait.values.reshape(-1, 1) - Eval_dfT.Trait.values.reshape(-1, 1).mean())**2).sum()
            R2EvalT.append( (1-(uVal/vVal)) )
            RMSEEvalT.append( np.sqrt( uVal/Eval_dfT.shape[0] ) )

            # R2 and RMSE for valid set
            PredT = copy.deepcopy( ValiUn[Val_df.index.tolist(),ll] )
            ResT  = Val_df.Trait.values - np.array( PredT )
            uVal = (ResT**2).sum()
            vVal = ((Val_df.Trait.values.reshape(-1, 1) - Val_df.Trait.values.reshape(-1, 1).mean())**2).sum()
            R2ValT.append( (1-(uVal/vVal)) )
            RMSEValT.append( np.sqrt( uVal/Val_df.shape[0] ) )

            # Prediction for plots - Mapping traits
            for val in range( len(PredMapT) ):  # acq
                PredMapT[val][:,ll] = plsM.predict( SpecAllFull_sel_0_1_res[val][0,:,0:len(BRef)] ).reshape(SpecAllFull_sel_0_1_res[val].shape[1],)

        CoefBooF.append(CoefUn)
        InterBooF.append(InterBooT)
        PLSRFactF.append(PLSRFactT)
        R2EvalF.append(R2EvalT)
        RMSEEvalF.append(RMSEEvalT)
        R2ValF.append(R2ValT)
        RMSEValF.append(RMSEValT)
        TestUnF.append(TestUn)
        ValiUnF.append(ValiUn)
        OutDUnF.append( OutDUn if ii in [0,1] else [] )
        VIPOUnF.append(VIPOUn)
        PredMapF.append( PredMapT )
        sampleBooF.append( sampleBooT )

    CoefBooF_all.append(CoefBooF)
    InterBooF_all.append(InterBooF)
    PLSRFactF_all.append(PLSRFactF)
    R2EvalF_all.append(R2EvalF)
    RMSEEvalF_all.append(RMSEEvalF)
    R2ValF_all.append(R2ValF)
    RMSEValF_all.append(RMSEValF)
    TestUnF_all.append(TestUnF)
    ValiUnF_all.append(ValiUnF)
    OutDUnF_all.append(OutDUnF)
    VIPOUnF_all.append(VIPOUnF)
    PredMapF_all.append( PredMapF )
    sampleValF_all.append( sampleValF )
    sampleBooF_all.append( sampleBooF )

### Save or load the outputs

Reads the saved outputs. Set `SAVE_OUTPUTS` to save a new run first. *(Original cell 9.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs' )

versionRef = '01'

if( SAVE_OUTPUTS ):
    with open('CoefBooF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(CoefBooF_all, f)
    with open('InterBooF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(InterBooF_all, f)
    with open('PLSRFactF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(PLSRFactF_all, f)
    with open('R2EvalF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(R2EvalF_all, f)
    with open('RMSEEvalF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(RMSEEvalF_all, f)
    with open('R2ValF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(R2ValF_all, f)
    with open('RMSEValF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(RMSEValF_all, f)
    with open('TestUnF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(TestUnF_all, f)
    with open('ValiUnF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(ValiUnF_all, f)
    with open('OutDUnF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(OutDUnF_all, f)
    with open('VIPOUnF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(VIPOUnF_all, f)
    with open('PredMapF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(PredMapF_all, f)
    with open('sampleValF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(sampleValF_all, f)
    with open('sampleBooF_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(sampleBooF_all, f)
# load
with open('CoefBooF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    CoefBooF_allO = pickle.load(f)

with open('InterBooF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    InterBooF_allO = pickle.load(f)

with open('PLSRFactF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    PLSRFactF_allO = pickle.load(f)

with open('R2EvalF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    R2EvalF_allO = pickle.load(f)

with open('RMSEEvalF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    RMSEEvalF_allO = pickle.load(f)

with open('R2ValF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    R2ValF_allO = pickle.load(f)

with open('RMSEValF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    RMSEValF_allO = pickle.load(f)

with open('TestUnF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    TestUnF_allO = pickle.load(f)

with open('ValiUnF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    ValiUnF_allO = pickle.load(f)

with open('OutDUnF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    OutDUnF_allO = pickle.load(f)

with open('VIPOUnF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    VIPOUnF_allO = pickle.load(f)

with open('PredMapF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    PredMapF_allO = pickle.load(f)

with open('sampleValF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    sampleValF_allO = pickle.load(f)

with open('sampleBooF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    sampleBooF_allO = pickle.load(f)

### Summary

Calculates R2, RMSE and their spread, and summarizes the predictions and VIP. *(Original cell 10.)*

In [ ]:
TraitRef  = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]  # reduced version of the traits - since crude protein is estimated from N content

# Prepare data output formats
R2RMSEvalSumm   = []
R2RMSValSumm    = []
R2RMSEvalSummIqr  = []
R2RMSValSummIqr   = []
PredObs     = []
PredObsVali = []
PredObsOut  = []
PredVIP     = []
PredObsIqr     = []
PredObsValiIqr = []
PredObsOutIqr  = []
PredVIPIqr     = []
IQRSumm = []
for val in range( len(TestUnF_allO) ): # dataset
    R2RMSEvalSumm.append( np.full( (len(R2EvalF_allO),3), np.nan ) )
    R2RMSValSumm.append(  np.full( (len(R2EvalF_allO),3), np.nan ) )
    R2RMSEvalSummIqr.append( np.full( (len(R2EvalF_allO),3), np.nan ) )
    R2RMSValSummIqr.append(  np.full( (len(R2EvalF_allO),3), np.nan ) )
    PredObs.append( [] )
    PredObsVali.append( [] )
    PredObsOut.append( [] )
    PredVIP.append( [] )
    PredObsIqr.append( [] )
    PredObsValiIqr.append( [] )
    PredObsOutIqr.append( [] )
    PredVIPIqr.append( [] )
    IQRSumm.append( np.empty( (1,len(TraitRef)) ) )

modeRef = 'mean'

DateRef  = [ int(SpecAll_sel_0_1_res_Avg.Date.unique().tolist()[val]) for val in range(len(SpecAll_sel_0_1_res_Avg.Date.unique().tolist())) ]
DateRefY = np.unique( [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ] ).tolist()

# nbr of bands with data
waveRef    = SpecAll_sel_0_1_res_Avg.shape[1] - len(colRef)
waveRefRed = SpecAll_sel_0_1_res_Avg_treat.shape[1] - (len(colRefRed)+ ( np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1].max() - \
                                                                        (np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1].min()-1) if \
                                                                        len(np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1]) >0 else 0 ))

for ii in range(len(R2EvalF_allO)):  # dataset
    for jj in range( len(R2EvalF_allO[ii]) ):  # properties
        R2RMSEvalSumm[ii][jj,1] = np.nanmedian( R2EvalF_allO[ii][jj] )   if modeRef == 'med' else np.nanmean( R2EvalF_allO[ii][jj] )
        R2RMSEvalSumm[ii][jj,2] = np.nanmedian( RMSEEvalF_allO[ii][jj] ) if modeRef == 'med' else np.nanmean( RMSEEvalF_allO[ii][jj] )
        R2RMSEvalSumm[ii][jj,0] = scipy.stats.mode( PLSRFactF_allO[ii][jj] )[0].tolist()[0]
        R2RMSValSumm[ii][jj,1]  = np.nanmedian( R2ValF_allO[ii][jj] )    if modeRef == 'med' else np.nanmean( R2ValF_allO[ii][jj] )
        R2RMSValSumm[ii][jj,2]  = np.nanmedian( RMSEValF_allO[ii][jj] )  if modeRef == 'med' else np.nanmean( RMSEValF_allO[ii][jj] )
        R2RMSEvalSummIqr[ii][jj,1] = scipy.stats.iqr( R2EvalF_allO[ii][jj], nan_policy='omit' )   if modeRef == 'med' else np.nanstd( R2EvalF_allO[ii][jj] )
        R2RMSEvalSummIqr[ii][jj,2] = scipy.stats.iqr( RMSEEvalF_allO[ii][jj], nan_policy='omit' ) if modeRef == 'med' else np.nanstd( RMSEEvalF_allO[ii][jj] )
        R2RMSValSummIqr[ii][jj,1]  = scipy.stats.iqr( R2ValF_allO[ii][jj], nan_policy='omit' )    if modeRef == 'med' else np.nanstd( R2ValF_allO[ii][jj] )
        R2RMSValSummIqr[ii][jj,2]  = scipy.stats.iqr( RMSEValF_allO[ii][jj], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( RMSEValF_allO[ii][jj] )
        PredObs[ii].append(     np.nanmedian( TestUnF_allO[ii][jj], axis=1 ).tolist() if modeRef == 'med' else np.nanmean( TestUnF_allO[ii][jj], axis=1 ).tolist() )
        PredObsVali[ii].append( np.nanmedian( ValiUnF_allO[ii][jj], axis=1 ).tolist() if modeRef == 'med' else np.nanmean( ValiUnF_allO[ii][jj], axis=1 ).tolist() )
        if ii in [0,1]:
            PredObsOut[ii].append(  np.nanmedian( OutDUnF_allO[ii][jj], axis=1 ).tolist() if modeRef == 'med' else np.nanmean( OutDUnF_allO[ii][jj], axis=1 ).tolist() )
        PredObsIqr[ii].append(     scipy.stats.iqr( TestUnF_allO[ii][jj], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( TestUnF_allO[ii][jj], axis=1 ).tolist() )
        PredObsValiIqr[ii].append( scipy.stats.iqr( ValiUnF_allO[ii][jj], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( ValiUnF_allO[ii][jj], axis=1 ).tolist() )
        if ii in [0,1]:
            PredObsOutIqr[ii].append(  scipy.stats.iqr( OutDUnF_allO[ii][jj], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( ValiUnF_allO[ii][jj], axis=1 ).tolist() )

        # VIP values
        PredVIP[ii].append(    np.nanmedian( VIPOUnF_allO[ii][jj], axis=1 ).tolist() if modeRef == 'med' else np.nanmean( VIPOUnF_allO[ii][jj], axis=1 ).tolist() )
        PredVIPIqr[ii].append( scipy.stats.iqr( VIPOUnF_allO[ii][jj], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( VIPOUnF_allO[ii][jj], axis=1 ).tolist() )

        # get IQR for eac dataset
        if ii in [0,1]:
            PeriodRef = [ DateRefY[ii] ]
        else:
            PeriodRef = DateRefY
        if( jj == 0 ):
            DataP  = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
            GroupF = [ str(DataP.Replicate.tolist()[val]) + '_' + str(DataP.Treatment.tolist()[val]) for val in range(DataP.shape[0]) ]
            if ii in [0,1]:
                DataPOut  = SpecAll_sel_0_1_res_Avg.loc[ ~(SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
            else:
                DataPOut  = []
        else:
            DataP  = SpecAll_sel_0_1_res_Avg_treat.loc[ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
            GroupF = DataP.Treatment.tolist()
            if ii in [0,1]:
                DataPOut  = SpecAll_sel_0_1_res_Avg_treat.loc[~ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
            else:
                DataPOut  = []
        DataM_df    = DataP.loc[ :, (DataP.columns == TraitRef[jj]) | DataP.columns.isin(range(waveRefRed)) ]  # using always the reduced data
        if ii in [0,1]:
            DataPOut_df = DataPOut.loc[ :, (DataPOut.columns == TraitRef[jj]) | DataPOut.columns.isin(range(waveRefRed)) ]
        BRef = [ 'b' + str(range(waveRefRed)[val]) if range(waveRefRed)[val] > 9 else 'b0' + str(range(waveRefRed)[val]) for val in range(waveRefRed) ]
        DataM_df.columns = ['Trait'] + BRef
        DataM_df = DataM_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...
        if ii in [0,1]:
            DataPOut_df.columns = ['Trait'] + BRef
            DataPOut_df = DataPOut_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...
        IQRSumm[ii][0,jj] = scipy.stats.iqr( DataM_df.Trait, nan_policy='omit' )

## 4. Figures of the baseline models

### Settings

Settings read by the figures below. *(Original cell 11.)*

In [ ]:
TraitRef   = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]
compRefT = 'New_UpD.'

### Figure 2

R2 and RMSE relative to the IQR, per source and target dataset. *(Original cell 12.)*

In [ ]:
import seaborn as sns

matplotlib.rcParams['axes.linewidth'] = 0.8
matplotlib.rcParams['font.family']    = 'Arial'

TraitRefL2 = ['Biomass', 'N', 'Ntotal']

TraitRefL  = [ r'Dry matter (DM) $[\/dt\/ha^{-1}\/]$',
               r'N content $[\/\%\/in\/DM\/]$',
               r'N total $[\/kg\/ha^{-1}\/]$'  ]

YearRef = SpecAll_sel_0_1_res_Avg.Year.unique().tolist()
YearRef = YearRef + [YearRef]  # include period with all years...

DateRef = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ]

matplotlib.rcParams['axes.linewidth'] = 0.8
matplotlib.rcParams['legend.fontsize'] = 9
matplotlib.rcParams['legend.handleheight'] = 0.6
matplotlib.rcParams['legend.handlelength'] = 0.6
matplotlib.rcParams['legend.handletextpad'] = 0.2
matplotlib.rcParams['legend.labelspacing'] = 0.2
matplotlib.rcParams['legend.columnspacing'] = 0.65
matplotlib.rcParams['font.family'] = 'Arial'

fontS00=11
fontS0=11
fontS1=11
fontS2=12
fontS3=13
lwidth1=0.8
lwidth2=2.5

ParamToPRef = 'R2'

addColorBarT = True

saveOptT = True

# choose which reference to use for RMSE
RMSERefOpt = 'All'

# attention: should be the same as the one used for R2RMSValSummIqr
modeRef = 'mean'

cmapA = cmap=plt.get_cmap('Blues')
cmapB = plt.get_cmap( 'summer' )

for ii in pbar.progressbar( range( len(TraitRef) ) ):  # traits

    # get values to plot
    R2PlotT   = []
    RMSEPlotT = []
    ErPlotT   = []
    R2PlotOutT   = []
    RMSEPlotOutT = []
    ErPlotOutT   = []
    for jj in range(len(R2RMSEvalSumm)):  # dataset
        R2PlotT.append(   R2RMSValSumm[jj][ii,1] )
        if( RMSERefOpt == 'Ind' ): # individual reference values
            RMSEPlotT.append( R2RMSValSumm[jj][ii,2]/IQRSumm[jj][0,ii] )
            ErPlotT.append(   R2RMSValSummIqr[jj][ii,2]/IQRSumm[jj][0,ii] )
        else:
            RMSEPlotT.append( R2RMSValSumm[jj][ii,2]/IQRSumm[-1][0,ii] )
            ErPlotT.append(   R2RMSValSummIqr[jj][ii,2]/IQRSumm[-1][0,ii] )
        if( jj in [0,1] ):
            R2PlotOutTemp   = []
            RMSEPlotOutTemp = []
            for kk in range(OutDUnF_allO[jj][ii].shape[1]):  # boot repetition
                PeriodRef = [ YearRef[jj] ]
                if( ii == 0 ):  # trait
                    DataPOut  = SpecAll_sel_0_1_res_Avg.loc[ ~(SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
                else:
                    DataPOut  = SpecAll_sel_0_1_res_Avg_treat.loc[ ~(SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
                DataPOut_df = DataPOut.loc[ :, (DataPOut.columns == TraitRef[ii]) | DataPOut.columns.isin(range(waveRefRed)) ]
                DataPOut_df.columns = ['Trait'] + BRef
                DataPOut_df = DataPOut_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...
                PredOutT  = OutDUnF_allO[jj][ii][:,kk]
                ResT =  DataPOut_df.Trait - np.array(PredOutT)
                uVal = (ResT**2).sum()
                vVal = ((DataPOut_df.Trait.values.reshape(-1, 1) - DataPOut_df.Trait.values.reshape(-1, 1).mean())**2).sum()
                R2PlotOutTemp.append( (1-(uVal/vVal)) )
                RMSEPlotOutTemp.append( np.sqrt( uVal/DataPOut_df.shape[0] ) )
            R2PlotOutT.append(   np.nanmedian( R2PlotOutTemp )   if modeRef == 'med' else np.nanmean( R2PlotOutTemp ) )
            if( RMSERefOpt == 'Ind' ): # individual reference values
                RMSEPlotOutT.append( np.nanmedian( RMSEPlotOutTemp )/IQRSumm[jj][0,ii] if modeRef == 'med' else np.nanmean( RMSEPlotOutTemp )/IQRSumm[jj][0,ii] )
                ErPlotOutT.append(   (scipy.stats.iqr( RMSEPlotOutTemp, nan_policy='omit' )/2)/IQRSumm[jj][0,ii] if modeRef == 'med' else np.nanstd( RMSEPlotOutTemp )/IQRSumm[jj][0,ii] )
            else:
                RMSEPlotOutT.append( np.nanmedian( RMSEPlotOutTemp )/IQRSumm[-1][0,ii] if modeRef == 'med' else np.nanmean( RMSEPlotOutTemp )/IQRSumm[-1][0,ii] )
                ErPlotOutT.append(   (scipy.stats.iqr( RMSEPlotOutTemp, nan_policy='omit' )/2)/IQRSumm[-1][0,ii] if modeRef == 'med' else np.nanstd( RMSEPlotOutTemp )/IQRSumm[-1][0,ii] )

    # prep to plot R2
    ToPlotTemp    = np.empty((3,3))
    ToPlotTemp[:] = np.nan
    ToPlotTempD   = copy.deepcopy( ToPlotTemp )
    ToPlotTemp_RMSE  = copy.deepcopy( ToPlotTemp )
    ToPlotTempD_RMSE = copy.deepcopy( ToPlotTemp )
    ToPlotTemp_Er  = copy.deepcopy( ToPlotTemp )
    ToPlotTempD_Er = copy.deepcopy( ToPlotTemp )
    # R2
    R2PlotOutT = [ R2PlotOutT[val] if R2PlotOutT[val]>0 else 0.0 for val in range(len(R2PlotOutT)) ]
    for elmnt in range(ToPlotTemp.shape[0]):
        for elmnt1 in range(ToPlotTemp.shape[1]):
            if( elmnt == elmnt1 ):
                ToPlotTemp[elmnt,elmnt1]  = R2PlotT[elmnt]
                ToPlotTempD[elmnt,elmnt1] = R2PlotT[elmnt]
            elif( all([(elmnt==0), (elmnt1==1)]) ):
                ToPlotTemp[elmnt,elmnt1]  = R2PlotOutT[0]
                ToPlotTempD[elmnt,elmnt1] = np.nan
            elif( all([(elmnt==1), (elmnt1==0)]) ):
                ToPlotTemp[elmnt,elmnt1]  = R2PlotOutT[1]
                ToPlotTempD[elmnt,elmnt1] = np.nan
            else:
                ToPlotTemp[elmnt,elmnt1] = np.nan
                ToPlotTempD[elmnt,elmnt1] = np.nan
    # RMSE
    for elmnt in range(ToPlotTemp_RMSE.shape[0]):
        for elmnt1 in range(ToPlotTemp_RMSE.shape[1]):
            if( elmnt == elmnt1 ):
                ToPlotTemp_RMSE[elmnt,elmnt1]  = RMSEPlotT[elmnt]
                ToPlotTempD_RMSE[elmnt,elmnt1] = RMSEPlotT[elmnt]
            elif( all([(elmnt==0), (elmnt1==1)]) ):
                ToPlotTemp_RMSE[elmnt,elmnt1]  = RMSEPlotOutT[0]
                ToPlotTempD_RMSE[elmnt,elmnt1] = np.nan
            elif( all([(elmnt==1), (elmnt1==0)]) ):
                ToPlotTemp_RMSE[elmnt,elmnt1]  = RMSEPlotOutT[1]
                ToPlotTempD_RMSE[elmnt,elmnt1] = np.nan
            else:
                ToPlotTemp_RMSE[elmnt,elmnt1] = np.nan
                ToPlotTempD_RMSE[elmnt,elmnt1] = np.nan
    # Error
    for elmnt in range(ToPlotTemp_Er.shape[0]):
        for elmnt1 in range(ToPlotTemp_Er.shape[1]):
            if( elmnt == elmnt1 ):
                ToPlotTemp_Er[elmnt,elmnt1]  = ErPlotT[elmnt]
                ToPlotTempD_Er[elmnt,elmnt1] = ErPlotT[elmnt]
            elif( all([(elmnt==0), (elmnt1==1)]) ):
                ToPlotTemp_Er[elmnt,elmnt1]  = ErPlotOutT[0]
                ToPlotTempD_Er[elmnt,elmnt1] = np.nan
            elif( all([(elmnt==1), (elmnt1==0)]) ):
                ToPlotTemp_Er[elmnt,elmnt1]  = ErPlotOutT[1]
                ToPlotTempD_Er[elmnt,elmnt1] = np.nan
            else:
                ToPlotTemp_Er[elmnt,elmnt1] = np.nan
                ToPlotTempD_Er[elmnt,elmnt1] = np.nan

    # extend dimensions of matrices
    extFacT_col = 2
    extFacT_row = 2
    for val1 in range( ToPlotTemp.shape[0] ):  # row
        colExtVal  = []
        colExtValD = []
        colExtVal_RMSE  = []
        colExtValD_RMSE = []
        for val2 in range( ToPlotTemp.shape[1] ):  # col
            colExtVal  = colExtVal  + [ ToPlotTemp[val1, val2] ]*extFacT_col
            colExtValD = colExtValD + [ ToPlotTempD[val1, val2] ]*extFacT_col
            colExtVal_RMSE  = colExtVal_RMSE  + [ ToPlotTemp_RMSE[val1, val2] ]*extFacT_col
            colExtValD_RMSE = colExtValD_RMSE + [ ToPlotTempD_RMSE[val1, val2] ]*extFacT_col
        if( val1 == 0 ):
            ToPlotTempE  = np.tile(np.expand_dims( np.array(colExtVal), axis=0 ), (extFacT_row, 1))
            ToPlotTempDE = np.tile(np.expand_dims( np.array(colExtValD), axis=0 ), (extFacT_row, 1))
            ToPlotTempE_RMSE  = np.tile(np.expand_dims( np.array(colExtVal_RMSE), axis=0 ), (extFacT_row, 1))
            ToPlotTempDE_RMSE = np.tile(np.expand_dims( np.array(colExtValD_RMSE), axis=0 ), (extFacT_row, 1))
        else:
            ToPlotTempE  = np.append( ToPlotTempE, np.tile(np.expand_dims( np.array(colExtVal), axis=0 ),   (extFacT_row, 1)), axis=0 )
            ToPlotTempDE = np.append( ToPlotTempDE, np.tile(np.expand_dims( np.array(colExtValD), axis=0 ), (extFacT_row, 1)), axis=0 )
            ToPlotTempE_RMSE  = np.append( ToPlotTempE_RMSE, np.tile(np.expand_dims( np.array(colExtVal_RMSE), axis=0 ),   (extFacT_row, 1)), axis=0 )
            ToPlotTempDE_RMSE = np.append( ToPlotTempDE_RMSE, np.tile(np.expand_dims( np.array(colExtValD_RMSE), axis=0 ), (extFacT_row, 1)), axis=0 )

    # Plot matrix
    classLabT = ['2014', '2017', 'All Data']

    fig = pylab.figure( figsize=(2.9, 2.9) ) # 1:1
    ax  = fig.add_subplot( 1,1,1 )

    if( ParamToPRef == 'R2' ):
        vminRefT,  vmaxRefT = 0, 1
        imgT = ax.imshow( ToPlotTempE, interpolation='nearest', cmap=(cmapA if (ParamToPRef == 'R2') else cmapB), \
                          vmin=vminRefT, vmax=vmaxRefT, alpha=1 )
    else:
        vminRefT, vmaxRefT = 0, 5
        imgT = ax.imshow( ToPlotTempE_RMSE, interpolation='nearest', cmap=(cmapA if (ParamToPRef == 'R2') else cmapB), \
                          vmin=vminRefT, vmax=vmaxRefT, alpha=1 )

    ax.set_xlim( [-0.5, 5.5] )
    ax.set_ylim( [5.5, -0.5] )
    ax.set_xticks( [-0.5, 1.5, 3.5, 5.5], 'minor' )
    ax.set_yticks( [-0.5, 1.5, 3.5, 5.5], 'minor' )
    ax.set_xticks( [0.5, 2.5, 4.5] )
    ax.set_yticks( [0.5, 2.5, 4.5] )
    ax.set_xticklabels(classLabT, fontname = "Arial",  fontsize=fontS1, rotation=0 )
    ax.set_yticklabels(classLabT, fontname = "Arial",  fontsize=fontS1, rotation=90, va='center', ha='center' )

    ax.tick_params(axis='x', which='major', length=0, width=0, direction="in", labelsize=fontS1, pad=5)
    ax.tick_params(axis='y', which='major', length=0, width=0, direction="in", labelsize=fontS1, pad=8.0)
    ax.tick_params(axis='x', which='minor', length=2.5, width=lwidth1, direction="out", labelsize=fontS1, pad=5)
    ax.tick_params(axis='y', which='minor', length=2.5, width=lwidth1, direction="out", labelsize=fontS1, pad=8.0)

    fig.text( 0.015, 0.500, '[ Source Data ]', rotation=90,
              fontsize=fontS2, fontstyle='normal', va='center', ha='center', color='k')

    fig.text( 0.500, -0.005, '[ Target Data ]', rotation=0,
              fontsize=fontS2, fontstyle='normal', va='center', ha='center', color='k')

    # write metric values over cells
    offRefT   = 0.425 # 3 lines
    xCoordLab = [0.5, 2.5, 4.5]
    yCoordLab = [0.5, 2.5, 4.5]

    # R2
    fmt = '.2f'
    thresh  = vmaxRefT/2
    for elmntT in range(ToPlotTemp.shape[0]):  # line
        for elmntT1 in range(ToPlotTemp.shape[1]):  # col
            if( np.isnan(ToPlotTemp[elmntT, elmntT1]) == False ):
                ax.text(yCoordLab[ elmntT1 ], xCoordLab[ elmntT ]-offRefT,  format(ToPlotTemp[elmntT, elmntT1], fmt),
                         ha="center", va="center", fontsize=fontS0, fontstyle='italic',
                         color=("w" if ToPlotTemp[elmntT, elmntT1] > thresh else "k") )
            else:
                continue

    # RMSE
    fmt = '.2f'
    for elmntT in range(ToPlotTemp_RMSE.shape[0]):  # line
        for elmntT1 in range(ToPlotTemp_RMSE.shape[1]):  # col
            if( np.isnan(ToPlotTemp_RMSE[elmntT, elmntT1]) == False ):
                ax.text(yCoordLab[ elmntT1 ], xCoordLab[ elmntT ],  '[ ' + format(ToPlotTemp_RMSE[elmntT, elmntT1], fmt) + '  ',
                         ha="center", va="center", fontsize=fontS0, fontstyle='italic',
                         color=("w" if ToPlotTemp[elmntT, elmntT1] > thresh else "k") )
            else:
                continue

    # Error
    fmt = '.2f'
    for elmntT in range(ToPlotTemp_Er.shape[0]):  # line
        for elmntT1 in range(ToPlotTemp_Er.shape[1]):  # col
            if( np.isnan(ToPlotTemp_Er[elmntT, elmntT1]) == False ):
                ax.text(yCoordLab[ elmntT1 ], xCoordLab[ elmntT ]+offRefT,  '$\pm$' + format(ToPlotTemp_Er[elmntT, elmntT1], fmt) + ' ]',
                         ha="center", va="center", fontsize=fontS0, fontstyle='italic',
                         color=("w" if ToPlotTemp[elmntT, elmntT1] > thresh else "k") )
            else:
                continue

    for edge, spine in ax.spines.items():
        spine.set_zorder(5)
        spine.set_linewidth(lwidth1)

    # add legend - color bar
    if( addColorBarT == True ):
        cax  = ax.figure.add_axes([0.920, 0, 0.030, 0.30 ])
        cbar = ax.figure.colorbar(imgT, cax=cax, orientation='vertical')
        cbar.ax.tick_params( direction='in', labelsize=fontS0, labelcolor='k', width=lwidth1, color='k')   # , length=lineW1
        cbar.set_ticks([vminRefT, vmaxRefT/2, vmaxRefT])
        cbar.outline.set_edgecolor('k')
        cbar.outline.set_linewidth(lwidth1)
        for val in range(len(cbar.ax.get_yticklabels())):
            cbar.ax.get_yticklabels()[val].set_rotation(60)
            cbar.ax.get_yticklabels()[val].set_horizontalalignment('left')
            cbar.ax.get_yticklabels()[val].set_verticalalignment('center')
        fig.text( 1.125, 0.075, '$R^{2}$' if (ParamToPRef == 'R2') else '$RMSE$', rotation=90,
                  fontsize=fontS1, fontstyle='normal', va='center', ha='center', color='k')

    # plot lines indicating acq date | division between classes (white grid)
    ax.grid(which="minor", color="w", linestyle='-', linewidth=lwidth2, zorder=0)

    ax.axvline( x=3.5, ymin=0, ymax=1, linestyle='solid', linewidth=lwidth1, color='k', alpha=1, zorder=3 )
    ax.axhline( y=3.5, xmin=0, xmax=1, linestyle='solid', linewidth=lwidth1, color='k', alpha=1, zorder=3 )

    ax.axvline( x=1.5, ymin=1-(4/6), ymax=1, linestyle='solid', linewidth=lwidth1, color='k', alpha=1, zorder=3 )
    ax.axhline( y=1.5, xmin=0,       xmax=4/6, linestyle='solid', linewidth=lwidth1, color='k', alpha=1, zorder=3 )

    # save figure
    if( saveOptT == True ):
        rootDirRefT = r'<PROJECT_ROOT>\1_JypiterLabProcessing\4_Figures'
        os.chdir( rootDirRefT )
        if( addColorBarT == True ):
            plt.savefig('confMat_valid_singleSeason_' + str(TraitRefL2[ii]) + '_' + str(ParamToPRef) + '_leg.png',
                        dpi=600, facecolor='w', edgecolor='w',
                        orientation='portrait', papertype=None, format=None,
                        transparent=False, bbox_inches='tight', pad_inches=0.1)
        else:
            plt.savefig('confMat_valid_singleSeason_' + str(TraitRefL2[ii]) + '_' + str(ParamToPRef) + '.png',
                        dpi=600, facecolor='w', edgecolor='w',
                        orientation='portrait', papertype=None, format=None,
                        transparent=False, bbox_inches='tight', pad_inches=0.1)

### Figure 6, reference panels

Sample-wise predictions of the model trained on all dates. *(Original cell 13.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\DataProcessing\1_Script_source' )
import FakeParulaPalette as FakeParulaF
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.graphics.regressionplots import abline_plot
from statsmodels.sandbox.regression.predstd import wls_prediction_std
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator, AutoLocator, MaxNLocator)
minorLocator = AutoMinorLocator()
majorLocator = MaxNLocator(4)
from decimal import Decimal

TraitRefL  = [ r'Dry matter $[\/dt\/\/ha^{-1}\/]$',
               r'N content $[\/\%\/in\/DM\/]$',
               r'N total $[\/kg\/\/ha^{-1}\/]$'  ]

fontS0 = 9
fontS1 = 10
fontS2 = 11
fontS3 = 12

FontSizeG = 10
FontSizeT = 11

matplotlib.rcParams['axes.linewidth']  = 0.8
matplotlib.rcParams['font.family']     = 'Arial'
matplotlib.rcParams['legend.fontsize'] = 8
matplotlib.rcParams['legend.handlelength']   = 0.8

MajorTicksLength = 3.5
MinorTicksLength = 2.2
errorLineWidth   = 1.2

lwidth =  0.6

TraitRef    = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]  # reduced version of the traits - since crude protein is estimated from N content

YLimAllObs = [ [[0, 60], [0.90, 3.90], [5, 105]],
               [[0, 60], [0.90, 3.90], [5, 105]],
               [[0, 60], [0.90, 3.90], [5, 105]] ]

YLimAllObs_new = [ [[0, 60], [0.90, 3.90], [-15, 105]],
                   [[0, 60], [0.90, 3.90], [-15, 105]],
                   [[0, 60], [0.90, 3.90], [-15, 105]] ]

axisIntDistT   = [ [20, 0.75, 25], [20, 0.75, 25], [20, 0.75, 25] ]
axisIntDisLenT = [ [7, 5, 5], [7, 5, 5], [7, 5, 5] ]
axisIntRoundFT = [ ["%.1f", "%.2f", "%.1f"], ["%.1f", "%.2f", "%.1f"], ["%.1f", "%.2f", "%.1f"] ]

YearRef   = [ int(SpecAll_sel_0_1_res_Avg.Year.unique().tolist()[val]) for val in range(len(SpecAll_sel_0_1_res_Avg.Year.unique().tolist())) ]
YearRef = YearRef + [YearRef]  # include period with all years...

DateRef  = [ int(SpecAll_sel_0_1_res_Avg.Date.unique().tolist()[val]) for val in range(len(SpecAll_sel_0_1_res_Avg.Date.unique().tolist())) ]
DateRefY = [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ]

def flip(items, ncol):
    return itertools.chain(*[items[i::ncol] for i in range(ncol)])

cmap = matplotlib.cm.get_cmap('BrBG')
colorR = np.arange(0, 1+0.2, 0.2)
colorR = np.arange(0, 0.3+0.15, 0.15).tolist() + colorR[4:].tolist()

Parula_manual   = [ plt.matplotlib.cm.get_cmap('Greens')(0.70), plt.matplotlib.cm.get_cmap('Greens')(0.50), \
                   'blue', 'dodgerblue', 'lightskyblue']
colorR = list( range( len(Parula_manual) ) )

DateDOY = []
for elmnt in range(len(DateRef)):  # date
    dateT = pd.to_datetime(DateRef[elmnt], format='%Y%m%d')
    dateTRef = pd.Timestamp(year=dateT.year, month=1, day=1)
    DateDOY.append( (dateT - dateTRef).days + 1 )

modeRef = 'mean'

saveOpt = True

for ii in range( len(TestUnF_allO) ):  # Dataset
    if ii in [0,1]:
        PeriodRef   = [ YearRef[ii] ]
        AcqCalibRef = [ val for val in range(len(DateRefY)) if DateRefY[val] == YearRef[ii] ]
    else:
        PeriodRef   = YearRef[ii]
        AcqCalibRef = [ val for val in range(len(DateRefY)) if DateRefY[val] in YearRef[ii] ]

    for jj in range( len(TraitRef) ):  # trait
        if( jj == 0 ):
            DataP  = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
            GroupF = [ str(DataP.Replicate.tolist()[val]) + '_' + str(DataP.Treatment.tolist()[val]) for val in range(DataP.shape[0]) ]
            if( ii in [0,1] ):
                DataPOut  = SpecAll_sel_0_1_res_Avg.loc[ ~(SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
            else:
                DataPOut  = []
        else:
            DataP  = SpecAll_sel_0_1_res_Avg_treat.loc[ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
            GroupF = DataP.Treatment.tolist()
            if( ii in [0,1] ):
                DataPOut  = SpecAll_sel_0_1_res_Avg_treat.loc[~ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
            else:
                DataPOut  = []

        DataM_df = DataP.loc[ :, (DataP.columns == 'Date') | (DataP.columns == TraitRef[jj]) | DataP.columns.isin(range(waveRefRed)) ]  # using always the reduced data
        if ii in [0,1]:
            DataPOut_df = DataPOut.loc[ :, (DataP.columns == 'Date') | (DataPOut.columns == TraitRef[jj]) | DataPOut.columns.isin(range(waveRefRed)) ]
        BRef = [ 'b' + str(range(waveRefRed)[val]) if range(waveRefRed)[val] > 9 else 'b0' + str(range(waveRefRed)[val]) for val in range(waveRefRed) ]
        DataM_df.columns = ['Date', 'Trait'] + BRef
        DataM_df = DataM_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...
        if ii in [0,1]:
            DataPOut_df.columns = ['Date', 'Trait'] + BRef
            DataPOut_df = DataPOut_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...

        TestOT = TestUnF_allO[ii][jj]
        ValiOT = ValiUnF_allO[ii][jj]
        TestAvgOT = np.nanmedian( TestOT, axis=1 ).tolist() if modeRef == 'med' else np.nanmean( TestOT, axis=1 ).tolist()
        ValiAvgOT = np.nanmedian( ValiOT, axis=1 ).tolist() if modeRef == 'med' else np.nanmean( ValiOT, axis=1 ).tolist()

        TestIqrOT = scipy.stats.iqr( TestOT, nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( TestOT, axis=1 ).tolist()
        ValiIqrOT = scipy.stats.iqr( ValiOT, nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( ValiOT, axis=1 ).tolist()

        sampTestT = np.isnan( ValiAvgOT ).tolist()
        sampValiT = np.isnan( TestAvgOT ).tolist()

        fig, ax = plt.subplots(figsize=(2.3, 2.3) )
        fig.subplots_adjust(left=0.23, right=0.94 , bottom = 0.22, top = 0.90)

        DateRT = DataM_df.Date.unique().tolist()
        DataPlotColoT = []
        for val in range(len(DateRT)):  # date to plot
            TestAvgOTS = np.array(TestAvgOT)[((DataM_df.Date == DateRT[val]) & sampTestT)]
            TestIqrOTS = np.array(TestIqrOT)[((DataM_df.Date == DateRT[val]) & sampTestT)]
            TraitTTS    = DataM_df.loc[ (DataM_df.Date == DateRT[val]) & sampTestT, : ].Trait.tolist()

            ValiAvgOTS = np.array(ValiAvgOT)[((DataM_df.Date == DateRT[val]) & sampValiT)]
            ValiIqrOTS = np.array(ValiIqrOT)[((DataM_df.Date == DateRT[val]) & sampValiT)]
            TraitTVS   = DataM_df.loc[ (DataM_df.Date == DateRT[val]) & sampValiT, : ].Trait.tolist()

            # Compile complete dataset to be plot
            DataToPT      = np.append( np.array( [TestAvgOTS, TestIqrOTS, TraitTTS ] ).T, np.array( [ValiAvgOTS, ValiIqrOTS, TraitTVS] ).T, axis=0 )
            DataPlotColoT = DataPlotColoT + [Parula_manual[ AcqCalibRef[val] ]]*len(TestAvgOTS) + ['red']*len(ValiAvgOTS)  # colors to be used
            if( val == 0 ):
                DataPlotCompT = copy.deepcopy(DataToPT)
            else:
                DataPlotCompT = np.append( DataPlotCompT, DataToPT, axis=0 )

        # plot points randonly for better visualization
        IdxToPT = list( range( DataPlotCompT.shape[0] ) )
        random.seed( [132563, 23015, 76][jj] )
        IdxToPT = random.sample(IdxToPT, k=len(IdxToPT))
        for val in range( len(IdxToPT) ):
            ax.errorbar( DataPlotCompT[IdxToPT[val],0], DataPlotCompT[IdxToPT[val],2], xerr=DataPlotCompT[IdxToPT[val],1], fmt='none', color=DataPlotColoT[IdxToPT[val]],
                         mfc='None', markersize=4, linewidth =errorLineWidth, capsize=1.40, elinewidth=errorLineWidth, markeredgewidth=errorLineWidth,
                         label= 'All data', zorder=3 )

        ax.set_ylim( YLimAllObs_new[ii][jj][0],  YLimAllObs_new[ii][jj][-1] )
        ax.set_xlim( YLimAllObs_new[ii][jj][0],  YLimAllObs_new[ii][jj][-1] )

        XPlot = [ [ np.min( list(ax.get_xlim()) + YLimAllObs[ii][jj] ), np.max( list(ax.get_xlim()) + YLimAllObs[ii][jj] )], \
                [ np.min( list(ax.get_xlim()) + YLimAllObs[ii][jj] ), np.max( list(ax.get_xlim()) + YLimAllObs[ii][jj] )] ]
        ax.plot( XPlot[0], XPlot[1], 'k-',  linewidth=0.8, zorder=1 ) # 1:1 line

        ax.set_xlim( ax.get_xlim() )
        # Plot line passing over data used to fit the model...
        DataTL = pd.DataFrame( np.array([ DataP.loc[sampValiT, :].eval(TraitRef[jj]), np.array(ValiAvgOT)[sampValiT]]).T , columns = ['Obs', 'Pred'])
        results = smf.ols('Obs ~ Pred', data=DataTL).fit()
        MParam = results.params

        XPlot2 =  [ XPlot[0][0]-(0.1*XPlot[0][1]), XPlot[0][1]+(0.1*XPlot[0][1]) ]
        ax.plot( XPlot2, [ MParam.Intercept + MParam.Pred*XPlot2[val] for val in range(len(XPlot2)) ], '--', color='red', linewidth=0.8, zorder=1 )  # regression line

        # add R2 and RMSE corresponding to 100 models predicting the outside data
        fig.text( 0.2525, 0.930, '$R^{2}$ = ' + str("%.2f" % round(R2RMSValSumm[ii][jj,1], 3)) if R2RMSValSumm[ii][jj,1] > 0 else 'R-sqrd. = ' + str("%.2f" % round(0.00, 3)), \
                  va='center', rotation=0, size=FontSizeG, \
                  fontname = "Arial", fontstyle='normal', fontweight='normal', color='k')
        fig.text( 0.51, 0.930, ' ; RMSE = ' + str("%.2f" % round(R2RMSValSumm[ii][jj,2], 3)), va='center', rotation=0, size=FontSizeG, \
                  fontname = "Arial", fontstyle='normal', fontweight='normal', color='k')
        axisIntT = np.arange( YLimAllObs[ii][jj][0], YLimAllObs[ii][jj][-1]+axisIntDistT[ii][jj], axisIntDistT[ii][jj] ).round( 3 ).tolist()
        ax.set_yticks( axisIntT[ 0:axisIntDisLenT[ii][jj] ] )
        ax.set_yticklabels( [ str( axisIntRoundFT[ii][jj] % axisIntT[val]) for val in range(len( axisIntT[0:axisIntDisLenT[ii][jj]] )) ], fontname = "Arial",  fontsize=fontS2, ha='center' )
        ax.set_xticks(  axisIntT[0:axisIntDisLenT[ii][jj]] )
        ax.set_xticklabels( [ str( axisIntRoundFT[ii][jj] % axisIntT[val]) for val in range(len( axisIntT[0:axisIntDisLenT[ii][jj]] )) ], fontname = "Arial",  fontsize=fontS2, ha='center', rotation=45 )

        # set tick location automatically
        minorLocator1 = AutoMinorLocator()
        ax.plot( np.array(TestAvgOT)[ sampTestT].tolist() + np.array(ValiAvgOT)[ sampValiT].tolist(), \
                 DataP.loc[sampTestT, :].eval(TraitRef[jj]).tolist() + DataP.loc[sampValiT, :].eval(TraitRef[jj]).tolist(), 'none' )
        ax.yaxis.set_minor_locator(minorLocator1)
        minorLocator2 = AutoMinorLocator()
        ax.plot( np.array(TestAvgOT)[ sampTestT].tolist() + np.array(ValiAvgOT)[ sampValiT].tolist(), \
               DataP.loc[sampTestT, :].eval(TraitRef[jj]).tolist() + DataP.loc[sampValiT, :].eval(TraitRef[jj]).tolist(), 'none' )
        ax.xaxis.set_minor_locator(minorLocator2)

        ax.tick_params(axis='y', which='major', width=0.8, length=MajorTicksLength, direction ='in', labelsize=FontSizeG, pad=13)
        ax.tick_params(axis='y', which='minor', width=0.8, length=MinorTicksLength, direction ='in', labelsize=FontSizeG, pad=13)
        ax.tick_params(axis='x', which='major', width=0.8, length=MajorTicksLength, direction ='in', labelsize=FontSizeG, pad=2)
        ax.tick_params(axis='x', which='minor', width=0.8, length=MinorTicksLength, direction ='in', labelsize=FontSizeG, pad=2)

        ax.grid(b=1, which='major', axis='both', color='gray', alpha=0.4, linestyle='dotted', linewidth=0.8, zorder=0)
        ax.grid(b=1, which='minor', axis='both', color='gray', alpha=0.2, linestyle='dotted', linewidth=0.8, zorder=0)

        fig.text( 0.01, 0.55, TraitRefL[jj] + ' - ' + 'Observ.', va='center', rotation=90, size=FontSizeG, fontname = "Arial", fontstyle='normal', fontweight='normal')
        fig.text( 0.55, 0.04, TraitRefL[jj] + ' - ' + 'Pred.', ha='center', va='center', rotation=0, \
                size=FontSizeG, fontname = "Arial", fontstyle='normal', fontweight='normal')

        if( saveOpt == True ):
            os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\4_Figures' )
            plt.savefig("VI_ScatterYearAll_CalVal_ER_PLSR_" + str([2014, 2017, 'All'][ii]) + "_Best_" + TraitRef[jj] + '_' + compRefT + "png", dpi=600, facecolor='w', edgecolor='w',
                        orientation='portrait', format=None,
                        transparent=False, bbox_inches='tight', pad_inches=0.05) # , bbox_inches='tight'

## 5. Pixel-level predictions

### Prediction maps

Saves the predictions of the baseline models at pixel level, used for comparison in Figures 9 and 10. *(Original cell 16.)*

In [ ]:
TraitRef  = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]  # reduced version of the traits - since crude protein seems to be estimated from N content
TraitRefL = ['Dry matter [dt/ha]', 'N [%]', 'N [kg/ha]']

DateRef  = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = np.unique( [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ] ).tolist()
DateRefAll = [ ['20140515', '20141015'], ['20170510', '20170830', '20171026'] ]
acqRefAll = [ [1, 2], [1, 3, 4] ]

colRef     = ['Date', 'Year', 'Treatment', 'Replicate', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha']
colRefRed  = ['Date', 'Treatment', 'Year', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha']

# nbr of bands with data
waveRef    = SpecAll_sel_0_1_res_Avg.shape[1] - len(colRef)
waveRefRed = SpecAll_sel_0_1_res_Avg_treat.shape[1] - (len(colRefRed) + ( np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1].max() - \
                                                                         (np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1].min()-1) if \
                                                                         len(np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1]) > 0 else 0 ))  # last part >> remove NaN values

MetaFieldRefT = [ 'Id', 'Naam', 'Treat', 'Block', 'Replicate', 'TreatCode' ]

# buffer to be added to image patches before rotation - in pixels
buffRefT = 20

# indicate which data to use for mapping
SpecDataSelOpt = 'Full'

NoData_value = np.nan

modelDateRef = [0, -1]  # indicate which reference dataset to use ( for 2014 using date 0 - only same year as predcitions  )

for ii in range( len(DateRefY) ):  # dataset source |  year
    if( ii == 0 ): # only last date for each year
        DateRefSel = DateRef[ DateRef.index('20140515') ]  # v2 ( best linearity between observed and predicted values ; better to illustrate results )
    else:
        DateRefSel = DateRef[ DateRef.index('20171026') ]
    yearRef = int( DateRefY[ii] )
    acqRef  = int( acqRefAll[ii][ DateRefAll[ii].index(DateRefSel) ]  )
    if(yearRef == 2017):
        if(acqRef == 1):
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170509\HYMSY'  # Hyper - Acq 1
        else:
            if(acqRef == 3):
                HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170829\HYMSY\End Products_Manual'  # Hyper - Acq 3
            else:
                HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20171026\HYMSY\F1' # F1
    else:
        if(acqRef == 1):
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20140515\End Products manual_update'  # Hyper - Acq 1
        else:
            HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20141014\End Products\Update_manual_20190314'  # Hyper - Acq 3  (calib02)
    os.chdir( HomeDir )
    PatternUpload = '.\\' + '*Manual_A.bsq'
    HypFilesAll = glob.glob(PatternUpload)
    resRefImT = []
    for val in range( len(HypFilesAll) ):  # flight line
        ImgProcessGL = gdal.Open( HypFilesAll[val], gdal.GA_ReadOnly )  # open one band with gdal just to get metadata
        resRefImT.append( ImgProcessGL.GetGeoTransform()[1] )
    ProjectionD = ImgProcessGL.GetProjection()

    for jj in pbar.progressbar( range( len(PredMapF_allO[ii]) ) ):  # trait

        PredSelT     = PredMapF_allO[ modelDateRef[ii] ][jj][ DateRef.index(DateRefSel) ]  # only predictions made by model based on the complete dataset are considered for comparison
        MetaDataSelT = SpecAllFull[ DateRef.index(DateRefSel) ].loc[:, MetaFieldRefT ]
        BlockRefT    = np.sort( MetaDataSelT.Block.unique() ).tolist()

        for elmnt1 in range( len(BlockRefT) ):  # block
            RepRefT = np.sort( MetaDataSelT.loc[ (MetaDataSelT.Block == BlockRefT[elmnt1]), 'Replicate' ].unique() ).tolist()
            for elmnt2 in range( len(RepRefT) ):  # replicate
                TreatRefT = MetaDataSelT.loc[ (MetaDataSelT.Block     == BlockRefT[elmnt1]) & \
                                              (MetaDataSelT.Replicate == RepRefT[elmnt2]), 'TreatCode' ].unique().tolist()
                TreatRefT = np.sort( [ TreatRefT[val].split('.')[-1] for val in range( len(TreatRefT) ) ] ).tolist()
                for elmnt3 in range( len(TreatRefT) ):  # treatment
                    DataToPT    = PredSelT[ (MetaDataSelT.Block == BlockRefT[elmnt1]) & \
                                            (MetaDataSelT.Replicate == RepRefT[elmnt2]) & \
                                            (MetaDataSelT.TreatCode == str(RepRefT[elmnt2]) + '.' + str(TreatRefT[elmnt3])), : ]  # predictions

                    DataRefGeoT = SpecAll_geoFull[ DateRef.index(DateRefSel) ].loc[ (MetaDataSelT.Block == BlockRefT[elmnt1]) & \
                                                                                    (MetaDataSelT.Replicate == RepRefT[elmnt2]) & \
                                                                                    (MetaDataSelT.TreatCode == str(RepRefT[elmnt2]) + '.' + str(TreatRefT[elmnt3])), ]

                    xCoordTemp  =  DataRefGeoT.geometry.tolist()
                    xCoordTemp  =  [ xCoordTemp[val].coords.xy[0][0] for val in range( len(xCoordTemp) ) ]
                    yCoordTemp  =  DataRefGeoT.geometry.tolist()
                    yCoordTemp  =  [ yCoordTemp[val].coords.xy[1][0] for val in range( len(yCoordTemp) ) ]
                    xCoordT  =  [ min(xCoordTemp), max(xCoordTemp) ]
                    yCoordT  =  [ min(yCoordTemp), max(yCoordTemp) ]

                    # without buffer of -1 pixel left and upper sides
                    geoTransform     = [ xCoordT[0]-(min(resRefImT)/2), min(resRefImT), 0, yCoordT[1]-(min(resRefImT)/2), 0, -1*min(resRefImT) ]
                    geoTransformRstr = affine.Affine( min(resRefImT), 0, xCoordT[0]-(min(resRefImT)/2), \
                                                      0, -1*min(resRefImT), yCoordT[1]-(min(resRefImT)/2) )

                    x_res = int( np.floor( ((xCoordT[1]+min(resRefImT)) - (xCoordT[0]-min(resRefImT))) / min(resRefImT) ) )
                    y_res = int( np.floor( ((yCoordT[1]+min(resRefImT)) - (yCoordT[0]-min(resRefImT))) / min(resRefImT) ) )

                    # add buffer around image to avoid loss of data
                    geoTransform[0], geoTransform[3]  = geoTransform[0]-(buffRefT*min(resRefImT)), geoTransform[3]+(buffRefT*min(resRefImT))
                    geoTransformRstr = affine.Affine( min(resRefImT), 0, geoTransformRstr[2]-(buffRefT*min(resRefImT)), \
                                                      0, -1*min(resRefImT), geoTransformRstr[5]+(buffRefT*min(resRefImT)) )
                    x_res, y_res = x_res+(buffRefT*2), y_res+(buffRefT*2)  # multiply by 2 to consider both extremities in each direction

                    # remove predicted negative values and interpolate missing info

                    # get row col reference for all pxls
                    rowCoordT, colCoordT = rasterio.transform.rowcol( transform=geoTransformRstr, xs=xCoordTemp , ys=yCoordTemp )

                    # replace negative values
                    DataToPT_int    = np.empty( DataToPT.shape )
                    DataToPT_int[:] = np.nan
                    for val in range( DataToPT.shape[1] ):  # info layers - boot repetition
                        DataSelT     = DataToPT[:,val]
                        valKeepMaskT = (DataSelT >= 0)
                        DataSelTT = DataSelT[valKeepMaskT]
                        rowCoordTT, colCoordTT = np.array(rowCoordT)[valKeepMaskT], np.array(colCoordT)[valKeepMaskT]
                        PtsRefT  = np.concatenate( (np.expand_dims(rowCoordTT,1), np.expand_dims(colCoordTT,1)), axis=1 )
                        gridOutT = scipy.interpolate.griddata( points=PtsRefT, values=DataSelTT, \
                                                               xi=(rowCoordT, colCoordT), method='linear' )
                        DataToPT_int[:,val] = gridOutT

                    # summary of predictions
                    DataToPT_avg = np.nanmean( DataToPT_int, axis=1 ).reshape( (DataToPT_int.shape[0],1) )
                    DataToPT_std =  np.nanstd( DataToPT_int, axis=1 ).reshape( (DataToPT_int.shape[0],1) )
                    DataToPT     = np.concatenate( (DataToPT_avg, DataToPT_std), axis=1 )
                    if( any(np.isnan(DataToPT[:,0])) ):
                        DataRefGeoT = DataRefGeoT[ ~np.isnan(DataToPT[:,0]) ].copy(deep=True)
                        DataToPT    = copy.deepcopy( DataToPT[ ~np.isnan(DataToPT[:,0]) ] )
                    ColRefTemp  = [ 'avg', 'std' ]
                    DataToPGeoT = gpd.GeoDataFrame( data=DataToPT, columns=ColRefTemp, geometry=DataRefGeoT.geometry.tolist(), crs=DataRefGeoT.crs )
                    # save predictions as .shp
                    destFolderT = r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs\PLSR_pred_res\Plot-wise'
                    os.chdir( destFolderT )
                    DataToPGeoT.to_file( filename= str(DateRefSel) + '_' + TraitRef[jj] + '_' + str(SpecDataSelOpt) + '_Block0' + str(BlockRefT[elmnt1]) + '_Repl0' + str(RepRefT[elmnt2]) + '_Treat' + \
                                                   str(TreatRefT[elmnt3]) + '_' + 'PLSR_pred_orig_res' + '.shp' )

                    # create output image - n spectral bands
                    destFolderT = r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs\PLSR_pred_res\Plot-wise'
                    os.chdir( destFolderT )
                    output_raster = str(DateRefSel) + '_' + TraitRef[jj] + '_' + str(SpecDataSelOpt) + '_Block0' + str(BlockRefT[elmnt1]) + '_Repl0' + str(RepRefT[elmnt2]) + '_Treat' + \
                                    str(TreatRefT[elmnt3]) + '_' + 'PLSR_pred_orig_res' + '.tif'
                    driver   = gdal.GetDriverByName('GTiff')
                    out_data = driver.Create( os.path.join(destFolderT, output_raster), int(x_res+1), int(y_res+1), DataToPT.shape[1], gdalconst.GDT_Float64)
                    out_data.SetProjection( ProjectionD )
                    out_data.FlushCache()
                    out_data.SetGeoTransform( geoTransform )
                    out_data  = None

                    # gdal version - save images for each info layer
                    for mm in range( DataToPT.shape[1] ):  # info layer
                        out_data  = gdal.Open(output_raster, gdal.GA_Update)  # GA_ReadOnly
                        out_data.GetRasterBand( int(mm+1) ).SetNoDataValue( NoData_value )
                        out_data.GetRasterBand( int(mm+1) ).Fill( NoData_value )
                        source_ds    = ogr.Open( str(DateRefSel) + '_' + TraitRef[jj] + '_' + str(SpecDataSelOpt) + '_Block0' + str(BlockRefT[elmnt1]) + '_Repl0' + str(RepRefT[elmnt2]) + '_Treat' + \
                                                 str(TreatRefT[elmnt3]) + '_' + 'PLSR_pred_orig_res' + '.shp' )
                        dataInT_gdal = source_ds.GetLayer()
                        gdal.RasterizeLayer( out_data, [int(mm+1)], dataInT_gdal, options=[ 'ALL_TOUCHED=True', 'ATTRIBUTE='+ ColRefTemp[mm] ] )
                        out_data  = None
                        source_ds = None

## 6. Feature selection

### Parameters

Settings of the VIP confidence interval. *(Original cell 17.)*

In [ ]:
DateRef  = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = np.unique( [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ] ).tolist()

# indicate reference for the calculation of residuals
VIP_refMod = 'mean'  # 'full'  >>> 'mean' = average of boot VIP values  ;  'full' =  VIP for model with all samples

# indicate which value to use for N in the std formula (bootstrap repetitions or nbr of samples)
Nref = 'BootMod'

# significance level for Martens Uncertainty Test (MUT; Forina et al., 2004) - CI | t-test
coefSelThrsld = 0.1

### Models with the selected bands

Retains the bands whose lower VIP confidence limit is above one and refits the models with them. *(Original cell 18.)*

In [ ]:
# Variable Importance in Projection - formulation for VIP calculation   # https://github.com/scikit-learn/scikit-learn/issues/7050
def vip(model):
    t = model.x_scores_
    w = model.x_weights_
    q = model.y_loadings_
    p, h = w.shape
    vips = np.zeros((p,))
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)
    for i in range(p):
        weight  = np.array([ (w[i,j] / np.linalg.norm(w[:,j]))**2 for j in range(h) ])
        vips[i] = np.sqrt(p*(s.T @ weight)/total_s)
    return vips

# max and min nbr of factors to use
nbrFactMax_thrshld = 10
nbrFactMin         = 3

# model fitting - PLSR
TraitRef  = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]  # reduced version of the traits - since crude protein seems to be estimated from N content
TraitRefL = ['Dry matter [dt/ha]', 'N [%]', 'N [kg/ha]']

DateRef  = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = np.unique( [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ] ).tolist()

# get seed for training and test sets selection - calib and valid

seedAll = [325685, 73256841, 568]

# for CV repetition
BootSNbr   = 100 # nbr of resampling | repetitions
seedRepTmF = []
np.random.seed(seed=23577)
for elmnt in range(len(TraitRef)):
    seedRepTmF.append( np.random.uniform(low=0, high=1000000000, size=BootSNbr) )
    seedRepTmF[-1] = [ int(round(seedRepTmF[-1][val])) for val in range(len(seedRepTmF[-1])) ]

colRef     = ['Date', 'Year', 'Treatment', 'Replicate', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha']
colRefRed  = ['Date', 'Treatment', 'Year', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha']

# nbr of bands with data
waveRef    = SpecAll_sel_0_1_res_Avg.shape[1] - len(colRef)
waveRefRed = SpecAll_sel_0_1_res_Avg_treat.shape[1] - (len(colRefRed) + ( np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1].max() - \
                                                                         (np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1].min()-1) if \
                                                                         len(np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1]) > 0 else 0 ))  # last part >> remove NaN values

# indicate if dataset should be scaled before model fitting
ScaleOpt = True

CoefBooF_varSel_all   = []
InterBooF_varSel_all  = []
PLSRFactF_varSel_all  = []
R2EvalF_varSel_all    = []
RMSEEvalF_varSel_all  = []
R2ValF_varSel_all     = []
RMSEValF_varSel_all   = []
TestUnF_varSel_all    = []
ValiUnF_varSel_all    = []
OutDUnF_varSel_all    = []
VIPOUnF_varSel_all    = []
PredMapF_varSel_all   = []
sampleValF_varSel_all = []
sampleBooF_varSel_all = []
CIselVIPRefF_varSel_all  = []
VIPRetainedF_varSel_all  = []
CIOutF_varSel_all        = []
for ii in range( (len(DateRefY)+1) ):  # dataset (year) + combining all dates
    if ii in [0,1]:
        PeriodRef = [ DateRefY[ii] ]
    else:  # combining all dates
        PeriodRef = DateRefY
    CoefBooF   = []
    InterBooF  = []
    PLSRFactF  = []
    R2EvalF    = []
    RMSEEvalF  = []
    R2ValF     = []
    RMSEValF   = []
    TestUnF    = []
    ValiUnF    = []
    OutDUnF    = []
    VIPOUnF    = []
    PredMapF   = []
    sampleValF = []
    sampleBooF = []
    CIselVIPRefF   = []
    VIPRetainedF   = []
    CIOutF         = []
    for jj in range(len(TraitRef)):  # trait
        if( jj == 0 ):  # dataset to consider - average at plot level - Dry mass
            DataP  = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
            GroupF = [ str(DataP.Replicate.tolist()[val]) + '_' + str(DataP.Treatment.tolist()[val]) for val in range(DataP.shape[0]) ]
            if ii in [0,1]:  # dataset left outside
                DataPOut  = SpecAll_sel_0_1_res_Avg.loc[ ~(SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
            else:
                DataPOut  = []
        else:  # dataset to consider - average at treat level - Nitrogen
            DataP  = SpecAll_sel_0_1_res_Avg_treat.loc[ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
            GroupF = DataP.Treatment.tolist()
            if ii in [0,1]:  # dataset left outside
                DataPOut  = SpecAll_sel_0_1_res_Avg_treat.loc[~ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
            else:
                DataPOut  = []

        # reduce dataset to predictive and dependent variable only
        DataM_df = DataP.loc[ :, (DataP.columns == TraitRef[jj]) | DataP.columns.isin( range(waveRefRed) ) ]  # using always the reduced data
        BRef = [ 'b' + str(range(waveRefRed)[val]) if range(waveRefRed)[val] > 9 else 'b0' + str(range(waveRefRed)[val]) for val in range(waveRefRed) ]
        DataM_df.columns = ['Trait'] + BRef
        DataM_df = DataM_df.reset_index(drop=True)
        if( ii in [0,1] ):  # dataset left outside
            DataPOut_df = DataPOut.loc[ :, (DataPOut.columns == TraitRef[jj]) | DataPOut.columns.isin(range(waveRefRed)) ]
            DataPOut_df.columns = ['Trait'] + BRef
            DataPOut_df = DataPOut_df.reset_index(drop=True)

        # calib and valid datasets - 0.7/0.3
        gsTV = sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seedAll[ii])  # random nbr associated with dataset|year since this is the fctor that matters ;
                                                                                                               # e.g., for N the same samples are drawn for the same datatset|year
        trainIF_Oall = []
        testIF_Oall  = []
        for trainI, testI in gsTV.split(DataM_df.loc[:,BRef], DataM_df.Trait, GroupF):
            trainIF_Oall.append( trainI )
            testIF_Oall.append( testI )
        trainIF_Oall = trainIF_Oall[0].tolist()
        testIF_Oall  = testIF_Oall[0].tolist()

        # save reference to validation set - attention: based on row nbr not index
        sampleValF.append( testIF_Oall )

        # get calibration and validation sets
        Cal_df   = DataM_df.iloc[ trainIF_Oall,: ]
        GroupFTr = [ GroupF[val] for val in range(len(GroupF)) if val in trainIF_Oall ]
        Val_df   = DataM_df.iloc[ testIF_Oall,: ]
        GroupFTt = [ GroupF[val] for val in range(len(GroupF)) if val in testIF_Oall ]

        # Create arrays were to store predictions
        TestUn = np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan )  # samples in the calib x repetitions (boot)
        EvalUn = np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan )  # samples in the calib x repetitions (boot)
        ValiUn = np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan )
        if ii in [0,1]:
            OutDUn = np.full( (DataPOut_df.shape[0], len(seedRepTmF[jj])), np.nan )
        VIPOUn = np.full( (len(BRef), len(seedRepTmF[jj])), np.nan )
        CoefUn = np.full( (len(BRef), len(seedRepTmF[jj])), np.nan )

        # VIP values from model derived using all samples - full calibration dataset
        CoefUn_full  = np.full( (len(BRef), len(seedRepTmF[jj])), np.nan )
        VIPOUn_full  = np.full( (len(BRef), len(seedRepTmF[jj])), np.nan )
        PLSRFactTemp = PLSRFactF_allO[ii][jj]
        for elmnt in range( len(PLSRFactTemp) ):  # bootstrap repetition  ;  only difference is the number of factors used in PLSR model
            plsM_full = PLSRegression( n_components=PLSRFactTemp[elmnt], scale=ScaleOpt )
            plsM_full.fit( Cal_df.loc[:,BRef].values, Cal_df.loc[:,'Trait'].values.reshape( (Cal_df.loc[:,'Trait'].shape[0],1) ) )
            CoefUn_full[ :, elmnt] = [ plsM_full.coef_.tolist()[val][0] for val in range(len(plsM_full.coef_.tolist())) ]
            VIPOUn_full[ :, elmnt] = vip(plsM_full)

        # feature selection - VIP Conf. Intervals
        VIPUn_sel = VIPOUnF_allO[ii][jj]
        if( VIP_refMod == 'mean' ):  # mean for all boot repetitions
            VIP_meanT = np.nanmean( VIPUn_sel, axis=1 )
            VIP_meanT = np.repeat( np.expand_dims(VIP_meanT,1), VIPUn_sel.shape[1], axis=1 )
        else:  # using actual coefficients for complete dataset (changes due to nbr of PLSR factors...)
            VIP_meanT = copy.deepcopy( VIPOUn_full )
        VIP_residualT = ( VIPUn_sel - VIP_meanT )
        VIP_sqrResidualT = np.nansum( (VIP_residualT*VIP_residualT), axis=1 )
        if( Nref == 'BootMod' ):
            VIP_sdv_boot = np.sqrt( (VIP_sqrResidualT / (BootSNbr - 1)) )  # bootstrap
        else:
            VIP_sdv_boot = np.sqrt( (VIP_sqrResidualT / (Cal_df.shape[0] - 1)) )

        # use conf. interval to evaluate coefficient-wise significance - Afanador et al. (2014)
        CIselVIPRefTemp =  ['reject']*len(BRef)   # indication if conf. interval for the hypotehesis of averages equal 0 was accepted or rejected (significant coef.)
        CIOutTemp  = []
        quatileT   = 1-( (coefSelThrsld)/2 )
        for elmnt in range( VIPUn_sel.shape[0] ):  # feature
            if( Nref == 'BootMod' ):
                tQuantOT   = scipy.stats.t.ppf( q=quatileT, df=(BootSNbr-1) )  # t-statistic student
                CI_paramOT = ( np.nanmean( VIPUn_sel[elmnt, :] ) - tQuantOT*VIP_sdv_boot[elmnt], np.nanmean( VIPUn_sel[elmnt, :] ) + tQuantOT*VIP_sdv_boot[elmnt] )
            else:
                tQuantOT   = scipy.stats.t.ppf( q=quatileT, df=(Cal_dfUpT.shape[0]-1) )  # t-statistic student
                CI_paramOT = ( np.nanmean( VIPUn_sel[elmnt, :] ) - tQuantOT*VIP_sdv_boot[elmnt], np.nanmean( VIPUn_sel[elmnt, :] ) + tQuantOT*VIP_sdv_boot[elmnt] )
            CIselVIPRefTemp[ elmnt ] = ( 'reject' if (CI_paramOT[0] > 1) else 'accept' )
# indicate if hypotehsis of VIP equal 0 coef. should be rejected (significant coef) or accepted (insignificant coef)
            CIOutTemp.append( CI_paramOT )
        VIPRetainedF.append( np.where( (np.array(CIselVIPRefTemp) == 'reject') )[0].tolist() )

        # currently active features - update
        BRef_sel = np.array(BRef)[ VIPRetainedF[-1] ].tolist()

        # 'internal' repetition - bootstrap
        InterBooT  = []
        PLSRFactT  = []
        R2EvalT    = []
        RMSEEvalT  = []
        R2ValT     = []
        RMSEValT   = []
        sampleBooT = []
        for ll in pbar.progressbar( range(len(seedRepTmF[jj])) ):  # repetition time of bootstrap

            # get model fitting and test sets
            Cal_dfT   = Cal_df.copy(deep=True)
            GroupFTrT = copy.deepcopy(GroupFTr)
            sizeRef   =  int( (Cal_dfT.shape[0]*0.3333) / len(DataP.Date.unique().tolist()) )
            # sizeRef >> divide by nbr of acquistions since its grouped by this factor, therefore selected samples will be multiplied by this factor
            gsBoo   = sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size=sizeRef, random_state=seedRepTmF[jj][ll])
            for trainI, testI in gsBoo.split(Cal_dfT.loc[:,BRef_sel], Cal_dfT.Trait, GroupFTrT):
                testIF    = testI.tolist()
                testIFidx = Cal_df.loc[ Cal_dfT.iloc[testIF,:].index.tolist(), :].index.tolist()  # index needed to be used with drop

            # save reference to test set within each boot repetition - attention: based on row nbr not index
            sampleBooT.append( testIF )

            # update datasets
            Cal_dfT      = Cal_df.drop( labels = testIFidx )  # calibration set for the bootstrap repetition
            GroupFTrTidx = [ val for val in range(Cal_df.shape[0]) if Cal_df.index.tolist()[val] not in testIFidx ]
            GroupFTrT    = [ GroupFTr[val] for val in GroupFTrTidx ]
            Eval_dfT     = Cal_df.loc[ testIFidx, :]          # test set for the bootstrap repetition

            # fit PLSR model to calibration set corresponding to the bootstrap repetition
            RMSETO = []
            R2TO   = []
            nbrFactMax = np.min( [ min(Cal_dfT.shape), nbrFactMax_thrshld, len(BRef_sel) ] )  # maximun nbr of PLS factors should be <= nbr of samples
            for mm in range( (nbrFactMax-nbrFactMin)+1 ):  # nbr of PLS factors to try
                plsM      = PLSRegression( n_components=range(nbrFactMin, nbrFactMax+1, 1)[mm], scale=ScaleOpt )
                plsM.fit( Cal_dfT.loc[:,BRef_sel].values,  Cal_dfT.loc[:,'Trait'].values.reshape( (Cal_dfT.loc[:,'Trait'].shape[0],1) ) )
                PredEvalO = [ plsM.predict(Eval_dfT.loc[:,BRef_sel].values).tolist()[val][0] for val in range(len(plsM.predict(Eval_dfT.loc[:,BRef_sel].values).tolist())) ]
                ResT =  Eval_dfT.Trait - np.array(PredEvalO)
                uVal = (ResT**2).sum()
                vVal = ((Eval_dfT.Trait.values.reshape(-1, 1) - Eval_dfT.Trait.values.reshape(-1, 1).mean())**2).sum()
                R2TO.append( (1-(uVal/vVal)) )
                RMSETO.append( np.sqrt( uVal/Eval_dfT.shape[0] ) )

            # selected 'optimal' nbr of factors for boot repetition
            nbrFactSel = list(range(nbrFactMin, nbrFactMax+1, 1))[ np.where( (RMSETO == np.array(RMSETO).min()) )[0][0] ]
            PLSRFactT.append( nbrFactSel )
            plsM = PLSRegression( n_components=nbrFactSel, scale=ScaleOpt )
            plsM.fit( Cal_dfT.loc[:,BRef_sel].values,  Cal_dfT.loc[:,'Trait'].values.reshape( (Cal_dfT.loc[:,'Trait'].shape[0], 1) ) )

            # get predictions and model parameters
            TestUn[Cal_dfT.index.tolist(),ll] = plsM.predict(Cal_dfT.loc[:,BRef_sel].values).reshape( len(Cal_dfT.index.tolist()), )
            EvalUn[testIFidx,ll] =  plsM.predict(Eval_dfT.loc[:,BRef_sel].values).reshape( len(testIFidx), )
            ValiUn[Val_df.index.tolist(),ll]  = plsM.predict(Val_df.loc[:,BRef_sel].values).reshape( len(Val_df.index.tolist()), )
            if ii in [0,1]:
                OutDUn[:,ll] = plsM.predict( DataPOut_df.loc[:,BRef_sel].values ).reshape( DataPOut_df.shape[0], )
            VIPOUn[ VIPRetainedF[-1], ll] = vip(plsM)
            CoefUn[ VIPRetainedF[-1], ll] = [ plsM.coef_.tolist()[val][0] for val in range(len(plsM.coef_.tolist())) ]
            InterBooT.append( (plsM._y_mean - np.dot(plsM._x_mean , plsM.coef_)).tolist()[0] )  # https://stackoverflow.com/questions/41188410/how-obtain-the-intercept-of-the-pls-regression-sklearn

            # R2 and RMSE for calib set
            PredT = copy.deepcopy( EvalUn[testIFidx,ll] )
            ResT  = Eval_dfT.Trait.values - np.array( PredT )
            uVal = (ResT**2).sum()
            vVal = ((Eval_dfT.Trait.values.reshape(-1, 1) - Eval_dfT.Trait.values.reshape(-1, 1).mean())**2).sum()
            R2EvalT.append( (1-(uVal/vVal)) )
            RMSEEvalT.append( np.sqrt( uVal/Eval_dfT.shape[0] ) )

            # R2 and RMSE for valid set
            PredT = copy.deepcopy( ValiUn[Val_df.index.tolist(),ll] )
            ResT  = Val_df.Trait.values - np.array( PredT )
            uVal = (ResT**2).sum()
            vVal = ((Val_df.Trait.values.reshape(-1, 1) - Val_df.Trait.values.reshape(-1, 1).mean())**2).sum()
            R2ValT.append( (1-(uVal/vVal)) )
            RMSEValT.append( np.sqrt( uVal/Val_df.shape[0] ) )
        CoefBooF.append(CoefUn)
        InterBooF.append(InterBooT)
        PLSRFactF.append(PLSRFactT)
        R2EvalF.append(R2EvalT)
        RMSEEvalF.append(RMSEEvalT)
        R2ValF.append(R2ValT)
        RMSEValF.append(RMSEValT)
        TestUnF.append(TestUn)
        ValiUnF.append(ValiUn)
        OutDUnF.append( OutDUn if ii in [0,1] else [] )
        VIPOUnF.append(VIPOUn)
        PredMapF.append( PredMapT )
        sampleBooF.append( sampleBooT )
        CIselVIPRefF.append( CIselVIPRefTemp )
        CIOutF.append(  CIOutTemp )
    CoefBooF_varSel_all.append( CoefBooF )
    InterBooF_varSel_all.append( InterBooF )
    PLSRFactF_varSel_all.append( PLSRFactF )
    R2EvalF_varSel_all.append( R2EvalF )
    RMSEEvalF_varSel_all.append( RMSEEvalF )
    R2ValF_varSel_all.append( R2ValF )
    RMSEValF_varSel_all.append( RMSEValF )
    TestUnF_varSel_all.append( TestUnF )
    ValiUnF_varSel_all.append( ValiUnF )
    OutDUnF_varSel_all.append( OutDUnF )
    VIPOUnF_varSel_all.append( VIPOUnF )
    PredMapF_varSel_all.append( PredMapF )
    sampleValF_varSel_all.append( sampleValF )
    sampleBooF_varSel_all.append( sampleBooF )
    CIselVIPRefF_varSel_all.append( CIselVIPRefF )
    VIPRetainedF_varSel_all.append( VIPRetainedF )
    CIOutF_varSel_all.append( CIOutF )

### Save or load the outputs

Reads the saved outputs. Set `SAVE_OUTPUTS` to save a new run first. *(Original cell 19.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs' )

versionRef = '01'

if( SAVE_OUTPUTS ):
    with open('CoefBooF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(CoefBooF_varSel_all, f)
    with open('InterBooF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(InterBooF_varSel_all, f)
    with open('PLSRFactF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(PLSRFactF_varSel_all, f)
    with open('R2EvalF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(R2EvalF_varSel_all, f)
    with open('RMSEEvalF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(RMSEEvalF_varSel_all, f)
    with open('R2ValF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(R2ValF_varSel_all, f)
    with open('RMSEValF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(RMSEValF_varSel_all, f)
    with open('TestUnF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(TestUnF_varSel_all, f)
    with open('ValiUnF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(ValiUnF_varSel_all, f)
    with open('OutDUnF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(OutDUnF_varSel_all, f)
    with open('VIPOUnF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(VIPOUnF_varSel_all, f)
    with open('PredMapF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(PredMapF_varSel_all, f)
    with open('sampleValF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(sampleValF_varSel_all, f)
    with open('sampleBooF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(sampleBooF_varSel_all, f)
if( SAVE_OUTPUTS ):
    with open('CIselVIPRefF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(CIselVIPRefF_varSel_all, f)
    with open('VIPRetainedF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(VIPRetainedF_varSel_all, f)
    with open('CIOutF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'wb') as f:
        pickle.dump(CIOutF_varSel_all, f)
# load
with open('CoefBooF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    CoefBooF_varSel_allO = pickle.load(f)

with open('InterBooF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    InterBooF_varSel_allO = pickle.load(f)

with open('PLSRFactF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    PLSRFactF_varSel_allO = pickle.load(f)

with open('R2EvalF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    R2EvalF_varSel_allO = pickle.load(f)

with open('RMSEEvalF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    RMSEEvalF_varSel_allO = pickle.load(f)

with open('R2ValF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    R2ValF_varSel_allO = pickle.load(f)

with open('RMSEValF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    RMSEValF_varSel_allO = pickle.load(f)

with open('TestUnF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    TestUnF_varSel_allO = pickle.load(f)

with open('ValiUnF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    ValiUnF_varSel_allO = pickle.load(f)

with open('OutDUnF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    OutDUnF_varSel_allO = pickle.load(f)

with open('VIPOUnF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    VIPOUnF_varSel_allO = pickle.load(f)

with open('PredMapF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    PredMapF_varSel_allO = pickle.load(f)

with open('VIPRetainedF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    VIPRetainedF_varSel_allO = pickle.load(f)

with open('sampleBooF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    sampleBooF_varSel_allO = pickle.load(f)
with open('CIselVIPRefF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    CIselVIPRefF_varSel_allO = pickle.load(f)

with open('VIPRetainedF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    VIPRetainedF_varSel_allO = pickle.load(f)

with open('CIOutF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    CIOutF_varSel_allO = pickle.load(f)

### Summary

Calculates R2, RMSE and their spread after feature selection. *(Original cell 20.)*

In [ ]:
TraitRef  = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]  # reduced version of the traits - since crude protein is estimated from N content

# Prepare data output formats
R2RMSEvalSumm_varSel   = []
R2RMSValSumm_varSel    = []
R2RMSEvalSummIqr_varSel  = []
R2RMSValSummIqr_varSel   = []
PredObs_varSel     = []
PredObsVali_varSel = []
PredObsOut_varSel  = []
PredVIP_varSel     = []
PredObsIqr_varSel     = []
PredObsValiIqr_varSel = []
PredObsOutIqr_varSel  = []
PredVIPIqr_varSel     = []
IQRSumm_varSel = []
for val in range( len(TestUnF_allO) ): # dataset
    R2RMSEvalSumm_varSel.append( np.full( (len(R2EvalF_allO),3), np.nan ) )
    R2RMSValSumm_varSel.append(  np.full( (len(R2EvalF_allO),3), np.nan ) )
    R2RMSEvalSummIqr_varSel.append( np.full( (len(R2EvalF_allO),3), np.nan ) )
    R2RMSValSummIqr_varSel.append(  np.full( (len(R2EvalF_allO),3), np.nan ) )
    PredObs_varSel.append( [] )
    PredObsVali_varSel.append( [] )
    PredObsOut_varSel.append( [] )
    PredVIP_varSel.append( [] )
    PredObsIqr_varSel.append( [] )
    PredObsValiIqr_varSel.append( [] )
    PredObsOutIqr_varSel.append( [] )
    PredVIPIqr_varSel.append( [] )
    IQRSumm_varSel.append( np.empty( (1,len(TraitRef)) ) )

modeRef = 'mean'

DateRef  = [ int(SpecAll_sel_0_1_res_Avg.Date.unique().tolist()[val]) for val in range(len(SpecAll_sel_0_1_res_Avg.Date.unique().tolist())) ]
DateRefY = np.unique( [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ] ).tolist()

# nbr of bands with data
waveRef    = SpecAll_sel_0_1_res_Avg.shape[1] - len(colRef)
waveRefRed = SpecAll_sel_0_1_res_Avg_treat.shape[1] - (len(colRefRed)+ ( np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1].max() - \
                                                                        (np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1].min()-1) if \
                                                                        len(np.where(SpecAll_sel_0_1_res_Avg_treat.isnull().values)[1]) >0 else 0 ))

for ii in range(len(R2EvalF_varSel_allO)):  # dataset
    for jj in range( len(R2EvalF_varSel_allO[ii]) ):  # traits

        R2RMSEvalSumm_varSel[ii][jj,1] = np.nanmedian( R2EvalF_varSel_allO[ii][jj] )   if modeRef == 'med' else np.nanmean( R2EvalF_varSel_allO[ii][jj] )
        R2RMSEvalSumm_varSel[ii][jj,2] = np.nanmedian( RMSEEvalF_varSel_allO[ii][jj] ) if modeRef == 'med' else np.nanmean( RMSEEvalF_varSel_allO[ii][jj] )
        R2RMSEvalSumm_varSel[ii][jj,0] = scipy.stats.mode( PLSRFactF_varSel_allO[ii][jj] )[0].tolist()[0]
        R2RMSValSumm_varSel[ii][jj,1]  = np.nanmedian( R2ValF_varSel_allO[ii][jj] )    if modeRef == 'med' else np.nanmean( R2ValF_varSel_allO[ii][jj] )
        R2RMSValSumm_varSel[ii][jj,2]  = np.nanmedian( RMSEValF_varSel_allO[ii][jj] )  if modeRef == 'med' else np.nanmean( RMSEValF_varSel_allO[ii][jj] )
        R2RMSEvalSummIqr_varSel[ii][jj,1] = scipy.stats.iqr( R2EvalF_varSel_allO[ii][jj], nan_policy='omit' )   if modeRef == 'med' else np.nanstd( R2EvalF_varSel_allO[ii][jj] )
        R2RMSEvalSummIqr_varSel[ii][jj,2] = scipy.stats.iqr( RMSEEvalF_varSel_allO[ii][jj], nan_policy='omit' ) if modeRef == 'med' else np.nanstd( RMSEEvalF_varSel_allO[ii][jj] )
        R2RMSValSummIqr_varSel[ii][jj,1]  = scipy.stats.iqr( R2ValF_varSel_allO[ii][jj], nan_policy='omit' )    if modeRef == 'med' else np.nanstd( R2ValF_varSel_allO[ii][jj] )
        R2RMSValSummIqr_varSel[ii][jj,2]  = scipy.stats.iqr( RMSEValF_varSel_allO[ii][jj], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( RMSEValF_varSel_allO[ii][jj] )
        PredObs_varSel[ii].append(     np.nanmedian( TestUnF_varSel_allO[ii][jj], axis=1 ).tolist() if modeRef == 'med' else np.nanmean( TestUnF_varSel_allO[ii][jj], axis=1 ).tolist() )
        PredObsVali_varSel[ii].append( np.nanmedian( ValiUnF_varSel_allO[ii][jj], axis=1 ).tolist() if modeRef == 'med' else np.nanmean( ValiUnF_varSel_allO[ii][jj], axis=1 ).tolist() )
        if ii in [0,1]:
            PredObsOut_varSel[ii].append(  np.nanmedian( OutDUnF_varSel_allO[ii][jj], axis=1 ).tolist() if modeRef == 'med' else np.nanmean( OutDUnF_varSel_allO[ii][jj], axis=1 ).tolist() )
        PredObsIqr_varSel[ii].append(     scipy.stats.iqr( TestUnF_varSel_allO[ii][jj], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( TestUnF_varSel_allO[ii][jj], axis=1 ).tolist() )
        PredObsValiIqr_varSel[ii].append( scipy.stats.iqr( ValiUnF_varSel_allO[ii][jj], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( ValiUnF_varSel_allO[ii][jj], axis=1 ).tolist() )
        if ii in [0,1]:
            PredObsOutIqr_varSel[ii].append(  scipy.stats.iqr( OutDUnF_varSel_allO[ii][jj], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( ValiUnF_varSel_allO[ii][jj], axis=1 ).tolist() )

        # VIP values
        PredVIP_varSel[ii].append(    np.nanmedian( VIPOUnF_varSel_allO[ii][jj], axis=1 ).tolist() if modeRef == 'med' else np.nanmean( VIPOUnF_varSel_allO[ii][jj], axis=1 ).tolist() )
        PredVIPIqr_varSel[ii].append( scipy.stats.iqr( VIPOUnF_varSel_allO[ii][jj], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( VIPOUnF_varSel_allO[ii][jj], axis=1 ).tolist() )

        # get IQR for eac dataset
        if ii in [0,1]:
            PeriodRef = [ DateRefY[ii] ]
        else:
            PeriodRef = DateRefY
        if( jj == 0 ):
            DataP  = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
            GroupF = [ str(DataP.Replicate.tolist()[val]) + '_' + str(DataP.Treatment.tolist()[val]) for val in range(DataP.shape[0]) ]
            if ii in [0,1]:
                DataPOut  = SpecAll_sel_0_1_res_Avg.loc[ ~(SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
            else:
                DataPOut  = []
        else:
            DataP  = SpecAll_sel_0_1_res_Avg_treat.loc[ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
            GroupF = DataP.Treatment.tolist()
            if ii in [0,1]:
                DataPOut  = SpecAll_sel_0_1_res_Avg_treat.loc[~ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
            else:
                DataPOut  = []
        DataM_df    = DataP.loc[ :, (DataP.columns == TraitRef[jj]) | DataP.columns.isin(range(waveRefRed)) ]  # using always the reduced data
        if ii in [0,1]:
            DataPOut_df = DataPOut.loc[ :, (DataPOut.columns == TraitRef[jj]) | DataPOut.columns.isin(range(waveRefRed)) ]
        BRef = [ 'b' + str(range(waveRefRed)[val]) if range(waveRefRed)[val] > 9 else 'b0' + str(range(waveRefRed)[val]) for val in range(waveRefRed) ]
        DataM_df.columns = ['Trait'] + BRef
        DataM_df = DataM_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...
        if ii in [0,1]:
            DataPOut_df.columns = ['Trait'] + BRef
            DataPOut_df = DataPOut_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...
        IQRSumm_varSel[ii][0,jj] = scipy.stats.iqr( DataM_df.Trait, nan_policy='omit' )

### Figure 8, reference panels

Sample-wise predictions of the model trained on all dates, using the selected bands. *(Original cell 21.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\DataProcessing\1_Script_source' )
import FakeParulaPalette as FakeParulaF
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.graphics.regressionplots import abline_plot
from statsmodels.sandbox.regression.predstd import wls_prediction_std
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator, AutoLocator, MaxNLocator)
minorLocator = AutoMinorLocator()
majorLocator = MaxNLocator(4)
from decimal import Decimal

TraitRefL  = [ r'Dry matter $[\/dt\/\/ha^{-1}\/]$',
               r'N content $[\/\%\/in\/DM\/]$',
               r'N total $[\/kg\/\/ha^{-1}\/]$'  ]

fontS0 = 9
fontS1 = 10
fontS2 = 11
fontS3 = 12
FontSizeG = 10
FontSizeT = 11

matplotlib.rcParams['axes.linewidth']  = 0.8
matplotlib.rcParams['font.family']     = 'Arial'
matplotlib.rcParams['legend.fontsize'] = 8
matplotlib.rcParams['legend.handlelength']   = 0.8

MajorTicksLength = 3.5
MinorTicksLength = 2.2
errorLineWidth   = 1.2

lwidth =  0.6

TraitRef    = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]  # reduced version of the traits - since crude protein is estimated from N content

YLimAllObs = [ [[0, 60], [0.90, 3.90], [5, 105]],
               [[0, 60], [0.90, 3.90], [5, 105]],
               [[0, 60], [0.90, 3.90], [5, 105]] ]
YLimAllObs_new = [ [[0, 60], [0.90, 3.90], [-15, 105]],
                   [[0, 60], [0.90, 3.90], [-15, 105]],
                   [[0, 60], [0.90, 3.90], [-15, 105]] ]

axisIntDistT   = [ [20, 0.75, 25], [20, 0.75, 25], [20, 0.75, 25] ]
axisIntDisLenT = [ [7, 5, 5], [7, 5, 5], [7, 5, 5] ]
axisIntRoundFT = [ ["%.1f", "%.2f", "%.1f"], ["%.1f", "%.2f", "%.1f"], ["%.1f", "%.2f", "%.1f"] ]

YearRef   = [ int(SpecAll_sel_0_1_res_Avg.Year.unique().tolist()[val]) for val in range(len(SpecAll_sel_0_1_res_Avg.Year.unique().tolist())) ]
YearRef = YearRef + [YearRef]  # include period with all years...

DateRef  = [ int(SpecAll_sel_0_1_res_Avg.Date.unique().tolist()[val]) for val in range(len(SpecAll_sel_0_1_res_Avg.Date.unique().tolist())) ]
DateRefY = [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ]

def flip(items, ncol):
    return itertools.chain(*[items[i::ncol] for i in range(ncol)])

cmap = matplotlib.cm.get_cmap('BrBG')
colorR = np.arange(0, 1+0.2, 0.2)
colorR = np.arange(0, 0.3+0.15, 0.15).tolist() + colorR[4:].tolist()

Parula_manual   = [ plt.matplotlib.cm.get_cmap('Greens')(0.70), plt.matplotlib.cm.get_cmap('Greens')(0.50), \
                   'blue', 'dodgerblue', 'lightskyblue']

DateDOY = []
for elmnt in range(len(DateRef)):  # date
    dateT = pd.to_datetime(DateRef[elmnt], format='%Y%m%d')
    dateTRef = pd.Timestamp(year=dateT.year, month=1, day=1)
    DateDOY.append( (dateT - dateTRef).days + 1 )

modeRef = 'mean'

saveOpt = True

for ii in range( len(TestUnF_varSel_allO) ):  # Dataset
    if ii in [0,1]:
        PeriodRef   = [ YearRef[ii] ]
        AcqCalibRef = [ val for val in range(len(DateRefY)) if DateRefY[val] == YearRef[ii] ]
    else:
        PeriodRef   = YearRef[ii]
        AcqCalibRef = [ val for val in range(len(DateRefY)) if DateRefY[val] in YearRef[ii] ]

    for jj in range(len(TraitRef)):  # trait
        if( jj == 0 ):
            DataP  = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
            GroupF = [ str(DataP.Replicate.tolist()[val]) + '_' + str(DataP.Treatment.tolist()[val]) for val in range(DataP.shape[0]) ]
            if( ii in [0,1] ):
                DataPOut  = SpecAll_sel_0_1_res_Avg.loc[ ~(SpecAll_sel_0_1_res_Avg.Year).astype(int).isin( PeriodRef ), :]
            else:
                DataPOut  = []
        else:
            DataP  = SpecAll_sel_0_1_res_Avg_treat.loc[ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
            GroupF = DataP.Treatment.tolist()
            if( ii in [0,1] ):
                DataPOut  = SpecAll_sel_0_1_res_Avg_treat.loc[~ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int).isin( PeriodRef ), :]
            else:
                DataPOut  = []

        DataM_df = DataP.loc[ :, (DataP.columns == 'Date') | (DataP.columns == TraitRef[jj]) | DataP.columns.isin(range(waveRefRed)) ]  # using always the reduced data
        if ii in [0,1]:
            DataPOut_df = DataPOut.loc[ :, (DataP.columns == 'Date') | (DataPOut.columns == TraitRef[jj]) | DataPOut.columns.isin(range(waveRefRed)) ]
        BRef = [ 'b' + str(range(waveRefRed)[val]) if range(waveRefRed)[val] > 9 else 'b0' + str(range(waveRefRed)[val]) for val in range(waveRefRed) ]
        DataM_df.columns = ['Date', 'Trait'] + BRef
        DataM_df = DataM_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...
        if ii in [0,1]:
            DataPOut_df.columns = ['Date', 'Trait'] + BRef
            DataPOut_df = DataPOut_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...

        TestOT = TestUnF_varSel_allO[ii][jj]
        ValiOT = ValiUnF_varSel_allO[ii][jj]
        TestAvgOT = np.nanmedian( TestOT, axis=1 ).tolist() if modeRef == 'med' else np.nanmean( TestOT, axis=1 ).tolist()
        ValiAvgOT = np.nanmedian( ValiOT, axis=1 ).tolist() if modeRef == 'med' else np.nanmean( ValiOT, axis=1 ).tolist()

        TestIqrOT = scipy.stats.iqr( TestOT, nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( TestOT, axis=1 ).tolist()
        ValiIqrOT = scipy.stats.iqr( ValiOT, nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( ValiOT, axis=1 ).tolist()

        sampTestT = np.isnan( ValiAvgOT ).tolist()
        sampValiT = np.isnan( TestAvgOT ).tolist()

        fig, ax = plt.subplots(figsize=(2.3, 2.3) )
        fig.subplots_adjust(left=0.23, right=0.94 , bottom = 0.22, top = 0.90)

        DateRT = DataM_df.Date.unique().tolist()
        DataPlotColoT = []
        for val in range(len(DateRT)):  # date to plot
            TestAvgOTS = np.array(TestAvgOT)[((DataM_df.Date == DateRT[val]) & sampTestT)]
            TestIqrOTS = np.array(TestIqrOT)[((DataM_df.Date == DateRT[val]) & sampTestT)]
            TraitTTS    = DataM_df.loc[ (DataM_df.Date == DateRT[val]) & sampTestT, : ].Trait.tolist()

            ValiAvgOTS = np.array(ValiAvgOT)[((DataM_df.Date == DateRT[val]) & sampValiT)]
            ValiIqrOTS = np.array(ValiIqrOT)[((DataM_df.Date == DateRT[val]) & sampValiT)]
            TraitTVS   = DataM_df.loc[ (DataM_df.Date == DateRT[val]) & sampValiT, : ].Trait.tolist()

            # Compile complete dataset to be plot
            DataToPT      = np.append( np.array( [TestAvgOTS, TestIqrOTS, TraitTTS] ).T, np.array( [ValiAvgOTS, ValiIqrOTS, TraitTVS] ).T, axis=0 )
            DataPlotColoT = DataPlotColoT + [Parula_manual[ AcqCalibRef[val] ]]*len(TestAvgOTS) + ['red']*len(ValiAvgOTS)  # colors to be used
            if( val == 0 ):
                DataPlotCompT = copy.deepcopy(DataToPT)
            else:
                DataPlotCompT = np.append( DataPlotCompT, DataToPT, axis=0 )

        # plot points randonly for better visualization
        IdxToPT = list( range( DataPlotCompT.shape[0] ) )
        random.seed( [132563, 23015, 76][jj] )
        IdxToPT = random.sample(IdxToPT, k=len(IdxToPT))
        for val in range( len(IdxToPT) ):
            ax.errorbar( DataPlotCompT[IdxToPT[val],0], DataPlotCompT[IdxToPT[val],2], xerr=DataPlotCompT[IdxToPT[val],1], fmt='none', color=DataPlotColoT[IdxToPT[val]],
                         mfc='None', markersize=4, linewidth =errorLineWidth, capsize=1.40, elinewidth=errorLineWidth, markeredgewidth=errorLineWidth,
                         label= 'All data', zorder=3 )

        ax.set_ylim( YLimAllObs_new[ii][jj][0],  YLimAllObs_new[ii][jj][-1] )
        ax.set_xlim( YLimAllObs_new[ii][jj][0],  YLimAllObs_new[ii][jj][-1] )

        XPlot = [ [ np.min( list(ax.get_xlim()) + YLimAllObs[ii][jj] ), np.max( list(ax.get_xlim()) + YLimAllObs[ii][jj] )], \
                [ np.min( list(ax.get_xlim()) + YLimAllObs[ii][jj] ), np.max( list(ax.get_xlim()) + YLimAllObs[ii][jj] )] ]
        ax.plot( XPlot[0], XPlot[1], 'k-',  linewidth=0.8 ) # 1:1 line

        ax.set_xlim( ax.get_xlim() )
        # Plot line passing over data used to fit the model...
        DataTL = pd.DataFrame( np.array([ DataP.loc[sampValiT, :].eval(TraitRef[jj]), np.array(ValiAvgOT)[sampValiT]]).T , columns = ['Obs', 'Pred'])
        results = smf.ols('Obs ~ Pred', data=DataTL).fit()
        MParam = results.params

        XPlot2 =  [ XPlot[0][0]-(0.1*XPlot[0][1]), XPlot[0][1]+(0.1*XPlot[0][1]) ]
        ax.plot( XPlot2, [ MParam.Intercept + MParam.Pred*XPlot2[val] for val in range(len(XPlot2)) ], '--', color='red', linewidth=0.8, zorder=2 )  # regression line

        # add R2 and RMSE corresponding to 100 models predicting the outside data
        fig.text( 0.2525, 0.930, '$R^{2}$ = ' + str("%.2f" % round(R2RMSValSumm_varSel[ii][jj,1], 3)) if R2RMSValSumm_varSel[ii][jj,1] > 0 else 'R-sqrd. = ' + str("%.2f" % round(0.00, 3)), \
                  va='center', rotation=0, size=FontSizeG, \
                  fontname = "Arial", fontstyle='normal', fontweight='normal', color='k')
        fig.text( 0.51, 0.930, ' ; RMSE = ' + str("%.2f" % round(R2RMSValSumm_varSel[ii][jj,2], 3)), va='center', rotation=0, size=FontSizeG, \
                  fontname = "Arial", fontstyle='normal', fontweight='normal', color='k')

        axisIntT = np.arange( YLimAllObs[ii][jj][0], YLimAllObs[ii][jj][-1]+axisIntDistT[ii][jj], axisIntDistT[ii][jj] ).round( 3 ).tolist()
        ax.set_yticks( axisIntT[ 0:axisIntDisLenT[ii][jj] ] )
        ax.set_yticklabels( [ str( axisIntRoundFT[ii][jj] % axisIntT[val]) for val in range(len( axisIntT[0:axisIntDisLenT[ii][jj]] )) ], fontname = "Arial",  fontsize=fontS2, ha='center' )
        ax.set_xticks(  axisIntT[0:axisIntDisLenT[ii][jj]] )
        ax.set_xticklabels( [ str( axisIntRoundFT[ii][jj] % axisIntT[val]) for val in range(len( axisIntT[0:axisIntDisLenT[ii][jj]] )) ], fontname = "Arial",  fontsize=fontS2, ha='center', rotation=45 )

        # set tick location automatically
        minorLocator1 = AutoMinorLocator()
        ax.plot( np.array(TestAvgOT)[ sampTestT].tolist() + np.array(ValiAvgOT)[ sampValiT].tolist(), \
               DataP.loc[sampTestT, :].eval(TraitRef[jj]).tolist() + DataP.loc[sampValiT, :].eval(TraitRef[jj]).tolist(), 'none' )
        ax.yaxis.set_minor_locator(minorLocator1)
        minorLocator2 = AutoMinorLocator()
        ax.plot( np.array(TestAvgOT)[ sampTestT].tolist() + np.array(ValiAvgOT)[ sampValiT].tolist(), \
               DataP.loc[sampTestT, :].eval(TraitRef[jj]).tolist() + DataP.loc[sampValiT, :].eval(TraitRef[jj]).tolist(), 'none' )
        ax.xaxis.set_minor_locator(minorLocator2)

        ax.tick_params(axis='y', which='major', width=0.8, length=MajorTicksLength, direction ='in', labelsize=FontSizeG, pad=13)
        ax.tick_params(axis='y', which='minor', width=0.8, length=MinorTicksLength, direction ='in', labelsize=FontSizeG, pad=13)
        ax.tick_params(axis='x', which='major', width=0.8, length=MajorTicksLength, direction ='in', labelsize=FontSizeG, pad=2)
        ax.tick_params(axis='x', which='minor', width=0.8, length=MinorTicksLength, direction ='in', labelsize=FontSizeG, pad=2)

        ax.grid(b=1, which='major', axis='both', color='gray', alpha=0.4, linestyle='dotted', linewidth=0.8)
        ax.grid(b=1, which='minor', axis='both', color='gray', alpha=0.2, linestyle='dotted', linewidth=0.8)

        fig.text( 0.01, 0.55, TraitRefL[jj] + ' - ' + 'Observ.', va='center', rotation=90, size=FontSizeG, fontname = "Arial", fontstyle='normal', fontweight='normal')
        fig.text( 0.55, 0.04, TraitRefL[jj] + ' - ' + 'Pred.', ha='center', va='center', rotation=0, \
                size=FontSizeG, fontname = "Arial", fontstyle='normal', fontweight='normal')

        if( saveOpt == True ):
            os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\4_Figures' )
            plt.savefig("VI_ScatterYearAll_CalVal_ER_PLSR_VarSel_" + str([2014, 2017, 'All'][ii]) + "_Best_" + TraitRef[jj] + '_' + compRefT + "png", dpi=600, facecolor='w', edgecolor='w',
                        orientation='portrait', format=None,
                        transparent=False, bbox_inches='tight', pad_inches=0.05) # , bbox_inches='tight'